### Carrier codes , Policy Number and Agent Code

In [1]:
# generate_policy_data.py
# Python 3.9+  (pandas optional but convenient)
import os
import csv
import random
import string
from typing import List, Dict

random.seed(20251008)

# --- Config ---
ROW_COUNT = 3000
OUT_PATH = "policy.csv"  # same file will be reused when you add columns later

# Realistic 4-char carrier abbreviations (not official, but believable)
CARRIER_CODES = [
    "TRAV",  # Travelers
    "HART",  # The Hartford
    "ALLS",  # Allstate
    "PRGS",  # Progressive
    "GEIC",  # GEICO
    "CHUB",  # Chubb
    "LIBM",  # Liberty Mutual
    "NATI",  # Nationwide
    "AMFA",  # American Family
    "FARM",  # Farmers
]
# Slight skew toward large carriers
CARRIER_WEIGHTS = [0.14, 0.10, 0.12, 0.14, 0.12, 0.08, 0.10, 0.08, 0.06, 0.06]

# Agent codes: typical 5–7 char numeric strings; we’ll use 6 digits with leading zeros
def make_agent_pool(n_agents: int = 300) -> List[str]:
    start = 120000
    return [f"{i:06d}" for i in range(start, start + n_agents)]

# PolicyNumber: AAA9999999 (avoid ambiguous letters like I, O, Q)
def gen_unique_policy_numbers(n: int) -> List[str]:
    letters = [c for c in string.ascii_uppercase if c not in {"I", "O", "Q"}]
    used = set()
    out = []
    while len(out) < n:
        prefix = "".join(random.choices(letters, k=3))
        suffix = f"{random.randint(0, 9_999_999):07d}"
        pol = f"{prefix}{suffix}"
        if pol not in used:
            used.add(pol)
            out.append(pol)
    return out

def generate_policy_rows(n: int) -> List[Dict[str, str]]:
    policy_numbers = gen_unique_policy_numbers(n)
    agents = make_agent_pool(300)

    rows = []
    for i in range(n):
        rows.append({
            "PolicyNumber": policy_numbers[i],
            "CarrierCode": random.choices(CARRIER_CODES, weights=CARRIER_WEIGHTS, k=1)[0],
            "AgentCode": random.choice(agents),
        })
    return rows

def write_csv(path: str, rows: List[Dict[str, str]]):
    fieldnames = ["PolicyNumber", "CarrierCode", "AgentCode"]
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)

# --- (For later) add columns to same CSV without changing row count ---
def append_columns_inplace(path: str, new_columns: Dict[str, str], order_after: str = None):
    """
    new_columns: dict of {col_name: default_value} to add if missing.
    order_after: insert new cols after this existing column (default: at end).
    """
    # Read all
    with open(path, "r", newline="", encoding="utf-8") as f:
        r = csv.DictReader(f)
        rows = list(r)
        existing_fields = r.fieldnames or []

    # Build new header
    new_fields = existing_fields.copy()
    insert_idx = (new_fields.index(order_after) + 1) if order_after in new_fields else len(new_fields)
    for col, default in new_columns.items():
        if col not in new_fields:
            new_fields.insert(insert_idx, col)
            insert_idx += 1

    # Fill defaults
    for row in rows:
        for col, default in new_columns.items():
            if col not in row or row[col] == "":
                row[col] = default

    # Write back
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=new_fields)
        w.writeheader()
        w.writerows(rows)

if __name__ == "__main__":
    rows = generate_policy_rows(ROW_COUNT)
    # ensure uniqueness of PK
    assert len({r["PolicyNumber"] for r in rows}) == ROW_COUNT, "Duplicate PolicyNumber generated."
    write_csv(OUT_PATH, rows)
    print(f"✓ Wrote {ROW_COUNT} rows to {OUT_PATH}")

    # Example (for later): append two new columns without changing rows
    # append_columns_inplace(OUT_PATH, {"PolicyStatus": "Active", "Channel": "Independent"}, order_after="AgentCode")


✓ Wrote 3000 rows to policy.csv


## Transaction Type

In [2]:
# add_transaction_type.py
import pandas as pd
import numpy as np
import random

random.seed(20251008)
np.random.seed(20251008)

in_path = "policy.csv"
out_path = "policy.csv"  # overwrite in place as requested

# Realistic mix for a current-term snapshot
values = ["New", "Renewal", "Endorsement", "Cancel", "Reinstate", "Rewrite"]
weights = [0.28, 0.60, 0.07, 0.03, 0.01, 0.01]  # sum to 1.0

df = pd.read_csv(in_path)

# If column already exists, do nothing (idempotent)
if "TransactionType" not in df.columns:
    # Generate values with the distribution above
    df["TransactionType"] = np.random.choice(values, size=len(df), p=weights)

    # Optional: tiny business sanity tweaks (keep rare events very rare)
    # e.g., if we ever model status later, we could align Cancel/Reinstate with it.

    # Save in place
    df.to_csv(out_path, index=False)
    print(f"✓ Added TransactionType to {out_path} (rows: {len(df)})")
else:
    print("TransactionType already exists; no changes made.")


✓ Added TransactionType to policy.csv (rows: 3000)


## Dates

In [21]:
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import random

random.seed(20251008)
np.random.seed(20251008)

IN_PATH = "policy.csv"
OUT_PATH = "policy.csv"

TERM_MONTHS = 12
MIN_TERM_START = datetime(2025, 4, 1)
MAX_TERM_END   = datetime(2026, 4, 1)
LATEST_ALLOWED_START = None  # computed below

OVERWRITE_DATES = True  # <-- flip to False to sanitize in place

def add_months(d: datetime, months: int) -> datetime:
    y = d.year + (d.month - 1 + months) // 12
    m = (d.month - 1 + months) % 12 + 1
    mdays = [31, 29 if (y%4==0 and (y%100!=0 or y%400==0)) else 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31][m-1]
    day = min(d.day, mdays)
    return d.replace(year=y, month=m, day=day)

def rand_date(start: datetime, end: datetime) -> datetime:
    delta = (end - start).days
    return start if delta <= 0 else start + timedelta(days=random.randint(0, delta))

LATEST_ALLOWED_START = add_months(MAX_TERM_END, -TERM_MONTHS)

df = pd.read_csv(IN_PATH)

# Ensure TransactionType exists
if "TransactionType" not in df.columns:
    vals = ["New","Renewal","Endorsement","Cancel","Reinstate","Rewrite"]
    wts  = [0.28, 0.60, 0.07, 0.03, 0.01, 0.01]
    df["TransactionType"] = np.random.choice(vals, size=len(df), p=wts)

def build_term_and_txn(ttype: str):
    peff = rand_date(MIN_TERM_START, LATEST_ALLOWED_START)
    pexp = add_months(peff, TERM_MONTHS)
    if ttype in ("New","Renewal","Rewrite"):
        teff = peff
    elif ttype == "Endorsement":
        teff = rand_date(peff + timedelta(days=1), pexp - timedelta(days=1))
    elif ttype == "Cancel":
        teff = rand_date(peff + timedelta(days=7), pexp)
        pexp = teff
    elif ttype == "Reinstate":
        teff = rand_date(peff + timedelta(days=7), pexp - timedelta(days=7))
    else:
        teff = peff
    return peff.date(), pexp.date(), teff.date()

def clamp_to_window(peff: pd.Timestamp, pexp: pd.Timestamp, ttype: str):
    """Sanitize existing values into the [2021-01-01 .. 2024-12-31] rule set."""
    # 1) Move effective into [MIN_TERM_START, LATEST_ALLOWED_START]
    if peff.to_pydatetime() < MIN_TERM_START:
        peff = pd.Timestamp(MIN_TERM_START.date())
    if peff.to_pydatetime() > LATEST_ALLOWED_START:
        peff = pd.Timestamp(LATEST_ALLOWED_START.date())
    # 2) Recompute expiration ~12 months later, capped at MAX_TERM_END
    pexp2 = pd.Timestamp(add_months(peff.to_pydatetime(), TERM_MONTHS).date())
    if pexp2.to_pydatetime() > MAX_TERM_END:
        pexp2 = pd.Timestamp(MAX_TERM_END.date())
    # 3) Ensure Cancel doesn’t extend past expiration
    if ttype == "Cancel":
        if "TransactionEffectiveDate" in df.columns:
            teff = pd.to_datetime(df.loc[idx, "TransactionEffectiveDate"])
            if teff.to_pydatetime() < peff.to_pydatetime():
                teff = peff
            pexp2 = min(pexp2, teff)
    return peff, pexp2

# Overwrite or sanitize
if OVERWRITE_DATES:
    eff, exp, txn = [], [], []
    for t in df["TransactionType"]:
        pe, px, tx = build_term_and_txn(t)
        eff.append(pe); exp.append(px); txn.append(tx)
    df["PolicyEffectiveDate"] = eff
    df["PolicyExpirationDate"] = exp
    df["TransactionEffectiveDate"] = txn
else:
    # Create missing columns first
    if "PolicyEffectiveDate" not in df.columns:
        df["PolicyEffectiveDate"] = pd.NaT
    if "PolicyExpirationDate" not in df.columns:
        df["PolicyExpirationDate"] = pd.NaT
    if "TransactionEffectiveDate" not in df.columns:
        df["TransactionEffectiveDate"] = pd.NaT

    for idx, t in df["TransactionType"].items():
        pe = pd.to_datetime(df.loc[idx, "PolicyEffectiveDate"], errors="coerce")
        px = pd.to_datetime(df.loc[idx, "PolicyExpirationDate"], errors="coerce")
        tx = pd.to_datetime(df.loc[idx, "TransactionEffectiveDate"], errors="coerce")

        if pd.isna(pe) or pd.isna(px):
            pe2, px2, tx2 = build_term_and_txn(t)
            df.loc[idx, "PolicyEffectiveDate"] = pe2
            df.loc[idx, "PolicyExpirationDate"] = px2
            if pd.isna(tx): df.loc[idx, "TransactionEffectiveDate"] = tx2
        else:
            pe2, px2 = clamp_to_window(pe, px, t)
            df.loc[idx, "PolicyEffectiveDate"] = pe2.date()
            df.loc[idx, "PolicyExpirationDate"] = px2.date()
            # keep txn inside the term if present
            if not pd.isna(tx):
                tx = max(pe2, min(tx, px2))
                df.loc[idx, "TransactionEffectiveDate"] = tx.date()

# Final guards
eff = pd.to_datetime(df["PolicyEffectiveDate"])
exp = pd.to_datetime(df["PolicyExpirationDate"])
assert exp.le(pd.Timestamp(MAX_TERM_END.date())).all(), "Found expiration > 2024-12-31"
assert exp.ge(eff).all(), "Found expiration before effective"

df.to_csv(OUT_PATH, index=False)
print(f"✓ Dates normalized within 2021–2024 → {OUT_PATH}")


✓ Dates normalized within 2021–2024 → policy.csv


In [9]:
import pandas as pd
df = pd.read_csv("policy.csv")
df["TermLengthMonths"] = 12
df.to_csv("policy.csv", index=False)
print("✓ Added TermLengthMonths=12 to all rows")


✓ Added TermLengthMonths=12 to all rows


## Claimant info

In [10]:
# add_policy_profile_columns.py
import pandas as pd
import numpy as np
import random
from faker import Faker

random.seed(20251008)
np.random.seed(20251008)
fake = Faker("en_US")

IN_PATH = "policy.csv"
OUT_PATH = "policy.csv"   # overwrite in place as we append new columns

# ------- Distributions / dictionaries -------
LOB_VALUES = ["Personal Auto", "Homeowners", "BOP", "WC", "Umbrella", "Package"]
LOB_WEIGHTS = [0.35, 0.25, 0.15, 0.10, 0.10, 0.05]  # tweak if you like

SUBLINE_MAP = {
    "Personal Auto": "Private Passenger Auto",
    "Homeowners": "HO3",
    "BOP": "Businessowners",
    "WC": "Workers Compensation",
    "Umbrella": "Personal Umbrella",   # could split into personal/commercial if needed
    "Package": "Commercial Package",
}

CONSTRUCTION_TYPES = ["Frame", "Masonry", "Masonry Veneer", "Joisted Masonry", "Non-Combustible", "Fire Resistive"]
CONSTR_WEIGHTS     = [0.35,    0.18,      0.12,             0.15,              0.12,               0.08]

def pick_state_abbr():
    # Use Faker's state_abbr for a US code like 'TX'
    return fake.state_abbr()

def make_risk_address(state_abbr: str) -> str:
    # Keep it short & clean, consistent with state
    street = fake.street_address()
    city = fake.city()
    zipc = fake.postcode()
    return f"{street}, {city}, {state_abbr} {zipc}"

def make_mailing_address(prefer_same: bool, risk_addr: str, risk_state: str) -> (str, str):
    if prefer_same:
        return risk_addr, risk_state
    # Different address (may be different state)
    st = pick_state_abbr()
    return make_risk_address(st), st

def name_and_email():
    first = fake.first_name()
    last = fake.last_name()
    # simple realistic email
    domain = random.choice(["example.com","mail.com","insmail.com","customer.net","policyhub.org"])
    email = f"{first}.{last}".lower() + "@" + domain
    return f"{first} {last}", email

def year_built():
    # Skew newer: triangular between 1950..2023 with mode ~2000
    y = int(round(random.triangular(1950, 2023, 2000)))
    return max(1900, min(y, 2024))

def protection_class():
    # ISO PPC 1..10, center around 5-7
    return int(np.clip(int(round(np.random.normal(6, 2))), 1, 10))

def blank_if(condition, value):
    return value if condition else ""

# ------- Load -------
df = pd.read_csv(IN_PATH)

# Quick sanity: keep PK unique
assert not df["PolicyNumber"].duplicated().any(), "Duplicate PolicyNumber detected."

# Add LineOfBusiness if not present
if "LineOfBusiness" not in df.columns:
    df["LineOfBusiness"] = np.random.choice(LOB_VALUES, size=len(df), p=LOB_WEIGHTS)

# Add Subline if missing
if "Subline" not in df.columns:
    df["Subline"] = df["LineOfBusiness"].map(SUBLINE_MAP).fillna("")

# InsuredName / Email
if "InsuredName" not in df.columns or "InsuredPrimaryEmail" not in df.columns:
    names, emails = [], []
    for _ in range(len(df)):
        n, e = name_and_email()
        names.append(n); emails.append(e)
    if "InsuredName" not in df.columns:
        df["InsuredName"] = names
    if "InsuredPrimaryEmail" not in df.columns:
        df["InsuredPrimaryEmail"] = emails

# Risk address + state
need_risk = ("RiskLocationAddress" not in df.columns) or ("State" not in df.columns)
if need_risk:
    risk_addresses, states = [], []
    for _ in range(len(df)):
        st = pick_state_abbr()
        risk_addresses.append(make_risk_address(st))
        states.append(st)
    if "RiskLocationAddress" not in df.columns:
        df["RiskLocationAddress"] = risk_addresses
    if "State" not in df.columns:
        df["State"] = states

# Mailing address (80% same as risk)
if "InsuredMailingAddress" not in df.columns:
    mailing, _states = [], []
    for i in range(len(df)):
        same = random.random() < 0.80
        addr, st = make_mailing_address(same, df["RiskLocationAddress"].iloc[i], df["State"].iloc[i])
        mailing.append(addr)
    df["InsuredMailingAddress"] = mailing

# Rated drivers & vehicles (only for Personal Auto)
if "RatedDriverCount" not in df.columns:
    rdc = []
    for lob in df["LineOfBusiness"]:
        if lob == "Personal Auto":
            # Typical ranges: 1–4 drivers
            rdc.append(int(np.clip(int(round(np.random.normal(2, 1))), 1, 5)))
        else:
            rdc.append("")  # blank when not applicable
    df["RatedDriverCount"] = rdc

if "VehicleCount" not in df.columns:
    vc = []
    for lob in df["LineOfBusiness"]:
        if lob == "Personal Auto":
            # Typical ranges: 1–3 vehicles
            vc.append(int(np.clip(int(round(np.random.normal(2, 1))), 1, 5)))
        else:
            vc.append("")  # blank when not applicable
    df["VehicleCount"] = vc

# DwellingYearBuilt (only for Homeowners)
if "DwellingYearBuilt" not in df.columns:
    yb = []
    for lob in df["LineOfBusiness"]:
        yb.append(year_built() if lob == "Homeowners" else "")
    df["DwellingYearBuilt"] = yb

# ConstructionType & ProtectionClass:
#   - Populate for Homeowners, BOP, Package. Leave blank for WC/Umbrella.
if "ConstructionType" not in df.columns:
    ct = []
    for lob in df["LineOfBusiness"]:
        if lob in ("Homeowners", "BOP", "Package"):
            ct.append(random.choices(CONSTRUCTION_TYPES, weights=CONSTR_WEIGHTS, k=1)[0])
        else:
            ct.append("")
    df["ConstructionType"] = ct

if "ProtectionClass" not in df.columns:
    ppc = []
    for lob in df["LineOfBusiness"]:
        if lob in ("Homeowners", "BOP", "Package"):
            ppc.append(protection_class())
        else:
            ppc.append("")
    df["ProtectionClass"] = ppc

# Save back
df.to_csv(OUT_PATH, index=False)
print(f"✓ Appended LOB/profile columns to {OUT_PATH} (rows: {len(df)})")


✓ Appended LOB/profile columns to policy.csv (rows: 3000)


## Numbers

In [23]:
# add_policy_limits_premiums.py
import pandas as pd
import numpy as np
import random
from datetime import datetime

random.seed(20251008)
np.random.seed(20251008)

IN_PATH = "policy.csv"
OUT_PATH = "policy.csv"  # overwrite in place per your workflow

# ---------- Helpers ----------
def clipf(x, lo, hi):
    return max(lo, min(hi, x))

def days_between(a: pd.Timestamp, b: pd.Timestamp) -> int:
    return max(0, (b - a).days)

def lognorm_amount(median: float, spread: float = 0.6, lo: float = None, hi: float = None):
    """
    Draw a positive amount with a log-normal-ish feel.
    'spread' is rough sigma in ln-space; 0.6 ~ moderate skew.
    """
    mu = np.log(median)  # approximate (ignoring exact median formula for simplicity)
    amt = float(np.random.lognormal(mean=mu, sigma=spread))
    if lo is not None: amt = max(lo, amt)
    if hi is not None: amt = min(hi, amt)
    return round(amt, 2)

# Common auto options
AUTO_BI = ["25/50", "50/100", "100/300", "250/500", "500/500"]
AUTO_BI_W = [0.10,     0.25,     0.45,       0.15,      0.05]

AUTO_PD = [25000, 50000, 100000, 250000, 500000]
AUTO_PD_W = [0.05, 0.20,   0.50,   0.20,   0.05]

DEDUCT = [250, 500, 1000, 2500]
DEDUCT_W = [0.15, 0.45, 0.30, 0.10]

BILLING = ["Direct Bill", "Agency Bill"]
BILLING_W = [0.70, 0.30]

# ---------- Load ----------
df = pd.read_csv(IN_PATH)

# Minimal guards
for col in ["PolicyEffectiveDate", "PolicyExpirationDate", "TransactionEffectiveDate"]:
    if col not in df.columns:
        raise ValueError(f"Missing required date column: {col}. Add dates before premiums.")
dates = ["PolicyEffectiveDate", "PolicyExpirationDate", "TransactionEffectiveDate"]
for c in dates:
    df[c] = pd.to_datetime(df[c], errors="coerce")

# Fallbacks if LOB/VehicleCount/ProtectionClass not present
if "LineOfBusiness" not in df.columns:
    df["LineOfBusiness"] = np.random.choice(
        ["Personal Auto", "Homeowners", "BOP", "WC", "Umbrella", "Package"],
        size=len(df),
        p=[0.50, 0.15, 0.05, 0.20, 0.05, 0.05]
    )
if "VehicleCount" not in df.columns:
    df["VehicleCount"] = np.where(df["LineOfBusiness"].eq("Personal Auto"),
                                  np.random.choice([1,2,3], size=len(df), p=[0.5,0.4,0.1]),
                                  np.nan)
if "ProtectionClass" not in df.columns:
    df["ProtectionClass"] = np.where(df["LineOfBusiness"].isin(["Homeowners","BOP","Package"]),
                                     np.clip(np.random.normal(6,2,size=len(df)).round().astype(int),1,10),
                                     np.nan)

# ---------- Auto-specific fields ----------
if "Limit_BI" not in df.columns:
    df["Limit_BI"] = np.where(
        df["LineOfBusiness"].eq("Personal Auto"),
        np.random.choice(AUTO_BI, size=len(df), p=AUTO_BI_W),
        ""
    )

if "Limit_PD" not in df.columns:
    df["Limit_PD"] = np.where(
        df["LineOfBusiness"].eq("Personal Auto"),
        np.random.choice(AUTO_PD, size=len(df), p=AUTO_PD_W),
        ""
    )

if "Deductible_Collision" not in df.columns:
    df["Deductible_Collision"] = np.where(
        df["LineOfBusiness"].eq("Personal Auto"),
        np.random.choice(DEDUCT, size=len(df), p=DEDUCT_W),
        ""
    )

if "Deductible_Comprehensive" not in df.columns:
    df["Deductible_Comprehensive"] = np.where(
        df["LineOfBusiness"].eq("Personal Auto"),
        np.random.choice(DEDUCT, size=len(df), p=DEDUCT_W),
        ""
    )

# ---------- TermPremium by LOB (realistic magnitudes) ----------
def term_premium_row(lob, veh_ct, ppc):
    if lob == "Personal Auto":
        base = 700 + 400*clipf((veh_ct if pd.notna(veh_ct) else 1), 1, 3)  # ~ $1.1k–$1.9k typical
        return lognorm_amount(median=base, spread=0.45, lo=300, hi=4000)
    elif lob == "Homeowners":
        adj = 1.0 + ((ppc if pd.notna(ppc) else 6) - 6) * 0.06  # PPC drives +/- ~30%
        return lognorm_amount(median=1500*adj, spread=0.55, lo=500, hi=7000)
    elif lob == "BOP":
        return lognorm_amount(median=4000, spread=0.7, lo=1500, hi=15000)
    elif lob == "WC":
        return lognorm_amount(median=5000, spread=0.9, lo=2000, hi=25000)
    elif lob == "Umbrella":
        return lognorm_amount(median=400, spread=0.5, lo=150, hi=2000)
    elif lob == "Package":
        return lognorm_amount(median=7000, spread=0.8, lo=3000, hi=30000)
    else:
        return lognorm_amount(median=2000, spread=0.7, lo=500, hi=20000)

if "TermPremium" not in df.columns:
    df["TermPremium"] = [
        term_premium_row(df.at[i,"LineOfBusiness"], df.at[i,"VehicleCount"], df.at[i,"ProtectionClass"])
        for i in range(len(df))
    ]

# ---------- WrittenPremium (transaction-basis, accounting-realistic) ----------
# Requirements: TransactionType, PolicyEffectiveDate, PolicyExpirationDate, TransactionEffectiveDate, TermPremium
for c in ["PolicyEffectiveDate","PolicyExpirationDate","TransactionEffectiveDate"]:
    df[c] = pd.to_datetime(df[c], errors="coerce")

def days_between(a: pd.Timestamp, b: pd.Timestamp) -> int:
    return 0 if (pd.isna(a) or pd.isna(b)) else max(0, (b - a).days)

def tx_written_amount(ttype, term_prem, eff, exp, txn):
    if ttype in ("New","Renewal","Rewrite"):
        return round(float(term_prem), 2)
    if ttype == "Endorsement":
        # write the delta only; scale to term size, allow +/-
        # ~N(0, 8% of term premium), clipped to +/- 25% of term
        sigma = 0.08 * float(term_prem)
        delta = float(np.random.normal(0.0, sigma))
        delta = clipf(delta, -0.25*float(term_prem), 0.25*float(term_prem))
        return round(delta, 2)
    if ttype == "Cancel":
        # negative unearned at cancel date
        term_days = max(1, days_between(eff, exp))
        earned_days = days_between(eff, min(txn, exp))
        earned_ratio = clipf(earned_days/term_days, 0.0, 1.0)
        unearned = float(term_prem) * (1.0 - earned_ratio)
        return round(-unearned, 2)
    if ttype == "Reinstate":
        return 0.00
    # fallback
    return round(float(term_prem), 2)

if "WrittenPremium" not in df.columns:
    df["WrittenPremium"] = [
        tx_written_amount(
            df.at[i,"TransactionType"],
            df.at[i,"TermPremium"],
            df.at[i,"PolicyEffectiveDate"],
            df.at[i,"PolicyExpirationDate"],
            df.at[i,"TransactionEffectiveDate"]
        )
        for i in range(len(df))
    ]

# ---------- EarnedPremium (pro-rata to AS_OF, capped to term) ----------
from datetime import datetime

AS_OF = datetime(2025, 12, 31)  # <-- pick portfolio as-of inside your 4/1/2025–4/1/2026 window

eff = pd.to_datetime(df["PolicyEffectiveDate"])
exp = pd.to_datetime(df["PolicyExpirationDate"])

# earn up to the earlier of AS_OF or expiration, but not before effective
asof = pd.Series(AS_OF, index=df.index).clip(lower=eff, upper=exp)

term_days = (exp - eff).dt.days.clip(lower=1)
elapsed   = (asof - eff).dt.days.clip(lower=0)
ratio     = (elapsed / term_days).clip(0, 1)

df["EarnedPremium"]   = (df["TermPremium"] * ratio).round(2)
df["UnearnedPremium"] = (df["TermPremium"] - df["EarnedPremium"]).round(2)

# ---------- BillingPlan ----------
if "BillingPlan" not in df.columns:
    df["BillingPlan"] = np.random.choice(BILLING, size=len(df), p=BILLING_W)


# Save back
df.to_csv(OUT_PATH, index=False)
print(f"✓ Added limits, deductibles, premiums, and billing plan → {OUT_PATH}")


✓ Added limits, deductibles, premiums, and billing plan → policy.csv


In [17]:
# add_cancel_reinstate_rewrite_edoc.py
import pandas as pd
import numpy as np

IN_PATH = "policy.csv"
OUT_PATH = "policy.csv"

df = pd.read_csv(IN_PATH)

# Parse dates we need
for c in ["PolicyEffectiveDate","PolicyExpirationDate","TransactionEffectiveDate"]:
    if c not in df.columns:
        raise ValueError(f"Missing {c}. Add dates before this step.")
    df[c] = pd.to_datetime(df[c], errors="coerce")

# Ensure TransactionType exists
if "TransactionType" not in df.columns:
    raise ValueError("Missing TransactionType. Add it before this step.")

# --- CancellationDate ---
if "CancellationDate" not in df.columns:
    # For Cancel transactions, set cancel date = transaction date; else blank
    cancel_mask = df["TransactionType"].eq("Cancel")
    canc_dates = pd.Series(pd.NaT, index=df.index)
    canc_dates.loc[cancel_mask] = df.loc[cancel_mask, "TransactionEffectiveDate"]
    # clamp inside term just in case
    canc_dates = canc_dates.where(canc_dates.isna(), canc_dates.clip(lower=df["PolicyEffectiveDate"], upper=df["PolicyExpirationDate"]))
    df["CancellationDate"] = canc_dates.dt.date

    # keep PolicyExpirationDate aligned to cancel date (snapshot convention)
    df.loc[cancel_mask, "PolicyExpirationDate"] = pd.to_datetime(df.loc[cancel_mask, "CancellationDate"])

# --- ReinstatementDate ---
if "ReinstatementDate" not in df.columns:
    rein_mask = df["TransactionType"].eq("Reinstate")
    rein_dates = pd.Series(pd.NaT, index=df.index)
    rein_dates.loc[rein_mask] = df.loc[rein_mask, "TransactionEffectiveDate"]
    # clamp inside term
    rein_dates = rein_dates.where(rein_dates.isna(), rein_dates.clip(lower=df["PolicyEffectiveDate"], upper=df["PolicyExpirationDate"]))
    df["ReinstatementDate"] = rein_dates.dt.date

# --- RewriteIndicator ---
if "RewriteIndicator" not in df.columns:
    df["RewriteIndicator"] = df["TransactionType"].eq("Rewrite")

# --- eDocIndicator (probability by LOB + billing plan) ---
# Base adoption by LOB (typical eDelivery take-rates; tweak if needed)
lob_base = {
    "Personal Auto": 0.90,
    "Homeowners":    0.85,
    "BOP":           0.70,
    "WC":            0.65,
    "Umbrella":      0.80,
    "Package":       0.70,
}
# Billing plan adjustment (Direct Bill tends to push eDocs higher)
bill_adj = {"Direct Bill": +0.05, "Agency Bill": -0.05}

if "eDocIndicator" not in df.columns:
    # Fallbacks if missing
    if "LineOfBusiness" not in df.columns:
        df["LineOfBusiness"] = "Personal Auto"
    if "BillingPlan" not in df.columns:
        df["BillingPlan"] = np.random.choice(["Direct Bill","Agency Bill"], p=[0.7,0.3], size=len(df))

    base = df["LineOfBusiness"].map(lob_base).fillna(0.75)
    adj  = df["BillingPlan"].map(bill_adj).fillna(0.0)
    prob = (base + adj).clip(0.05, 0.98)

    rng = np.random.default_rng(20251008)
    df["eDocIndicator"] = rng.uniform(0,1,len(df)) < prob

# Save
df.to_csv(OUT_PATH, index=False)
print(f"✓ Added CancellationDate, ReinstatementDate, RewriteIndicator, eDocIndicator → {OUT_PATH}")


✓ Added CancellationDate, ReinstatementDate, RewriteIndicator, eDocIndicator → policy.csv


## Commission table

In [24]:
# build_commission_from_policy.py
# Generate commission lines aligned to policy table (1-to-many realistic)
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

random.seed(20251008)
np.random.seed(20251008)

POLICY_PATH = "policy.csv"
OUT_PATH    = "commission.csv"

# ----------------- helpers -----------------
def month_end(d: pd.Timestamp) -> pd.Timestamp:
    if pd.isna(d):
        return pd.NaT
    # move to 1st of next month, then back 1 day
    y, m = d.year, d.month
    if m == 12:
        nxt = pd.Timestamp(year=y+1, month=1, day=1)
    else:
        nxt = pd.Timestamp(year=y, month=m+1, day=1)
    return nxt - pd.Timedelta(days=1)

def clamp(x, lo, hi):
    return max(lo, min(hi, x))

def pct_from_lob(lob: str) -> float:
    # Industry-sane street commission (base), bounded
    base = {
        "Personal Auto": (0.10, 0.02, 0.06, 0.14),  # mean, sd, lo, hi
        "Homeowners":    (0.13, 0.02, 0.08, 0.18),
        "BOP":           (0.15, 0.03, 0.08, 0.20),
        "WC":            (0.10, 0.02, 0.05, 0.15),
        "Umbrella":      (0.12, 0.02, 0.08, 0.18),
        "Package":       (0.14, 0.03, 0.08, 0.20),
    }
    mu, sd, lo, hi = base.get(lob, (0.12, 0.03, 0.06, 0.20))
    return round(clamp(float(np.random.normal(mu, sd)), lo, hi), 4)

def days_between(a: pd.Timestamp, b: pd.Timestamp) -> int:
    if pd.isna(a) or pd.isna(b):
        return 0
    return max(0, (b - a).days)

# ----------------- load policy -----------------
dfp = pd.read_csv(POLICY_PATH)
# required fields
req = ["PolicyNumber","CarrierCode","AgentCode","TransactionType",
       "PolicyEffectiveDate","PolicyExpirationDate","TransactionEffectiveDate","LineOfBusiness","TermPremium"]
for c in req:
    if c not in dfp.columns:
        raise ValueError(f"Missing required column in policy.csv: {c}")

# parse dates
for c in ["PolicyEffectiveDate","PolicyExpirationDate","TransactionEffectiveDate"]:
    dfp[c] = pd.to_datetime(dfp[c], errors="coerce")

# ----------------- build commission lines -----------------
rows = []

for i, r in dfp.iterrows():
    pol = r["PolicyNumber"]
    carr = r["CarrierCode"]
    agent = r["AgentCode"]
    lob = r["LineOfBusiness"]
    ttype = r["TransactionType"]
    eff = r["PolicyEffectiveDate"]
    exp = r["PolicyExpirationDate"]
    txn = r["TransactionEffectiveDate"]
    term_prem = float(r["TermPremium"])

    # Commission rate by LOB
    comm_pct = pct_from_lob(lob)

    # Basis (WrittenPremium) by transaction type (accounting-sane defaults)
    if ttype in ("New", "Renewal", "Rewrite"):
        basis = term_prem

    # --- inside the loop, for Endorsement ---
    elif ttype == "Endorsement":
        # write the delta only; +/- ~8% sigma, capped at +/-25% of term
        sigma = 0.08 * float(term_prem)
        delta = float(np.random.normal(0.0, sigma))
        delta = clamp(delta, -0.25 * float(term_prem), 0.25 * float(term_prem))
        basis = round(delta, 2)


    elif ttype == "Cancel":
        # return premium equal to unearned as of cancel date (negative)
        term_days = max(1, days_between(eff, exp))
        earned_days = days_between(eff, min(txn, exp))
        earned_ratio = clamp(earned_days / term_days, 0.0, 1.0)
        unearned = round(term_prem * (1 - earned_ratio), 2)
        basis = -abs(unearned)

    elif ttype == "Reinstate":
        # often $0 written for reinstatement (fees are separate)
        basis = 0.0

    else:
        basis = term_prem

    # StatementDate: month-end of transaction date (or effective if txn missing)
    base_date = txn if pd.notna(txn) else eff
    stmt_date = month_end(base_date) if pd.notna(base_date) else pd.NaT
    # DueDate ~ 15 days after statement
    due_date = (stmt_date + pd.Timedelta(days=15)) if pd.notna(stmt_date) else pd.NaT

    comm_amt = round(basis * comm_pct, 2)
    reversal = bool(comm_amt < 0)

    rows.append({
        "CarrierCode": carr,
        "AgentCode": agent,
        "StatementDate": None if pd.isna(stmt_date) else stmt_date.date(),
        "PolicyNumber": pol,
        "TransactionType": ttype if ttype in ["New","Renewal","Endorsement","Cancel","Audit","ReturnPremium","Reinstate","Rewrite"] else "New",
        "WrittenPremium": round(basis, 2),
        "CommissionPercent": comm_pct,
        "CommissionAmount": comm_amt,
        "DueDate": None if pd.isna(due_date) else due_date.date(),
        "ReversalIndicator": reversal,
    })

    # Optional: add occasional AUDIT lines for WC/BOP/Package (1-to-many realism)
    if lob in ("WC","BOP","Package") and random.random() < 0.08:
        # audit 15–90 days after expiration
        audit_when = exp + pd.Timedelta(days=int(np.random.uniform(15, 90)))
        stmt_aud = month_end(audit_when)
        due_aud  = stmt_aud + pd.Timedelta(days=15)
        delta_pct = np.random.uniform(-0.15, 0.15)
        audit_basis = round(term_prem * delta_pct, 2)
        audit_comm  = round(audit_basis * comm_pct, 2)
        rows.append({
            "CarrierCode": carr,
            "AgentCode": agent,
            "StatementDate": stmt_aud.date(),
            "PolicyNumber": pol,
            "TransactionType": "Audit",
            "WrittenPremium": audit_basis,
            "CommissionPercent": comm_pct,
            "CommissionAmount": audit_comm,
            "DueDate": due_aud.date(),
            "ReversalIndicator": bool(audit_comm < 0),
        })


# assemble & write
dfc = pd.DataFrame(rows, columns=[
    "CarrierCode","AgentCode","StatementDate","PolicyNumber","TransactionType",
    "WrittenPremium","CommissionPercent","CommissionAmount","DueDate","ReversalIndicator"
])

# align dtypes: CommissionPercent decimal, amounts 2dp, booleans are True/False
dfc["CommissionPercent"] = dfc["CommissionPercent"].astype(float).round(4)
dfc["WrittenPremium"] = dfc["WrittenPremium"].astype(float).round(2)
dfc["CommissionAmount"] = dfc["CommissionAmount"].astype(float).round(2)
dfc["ReversalIndicator"] = dfc["ReversalIndicator"].astype(bool)

dfc.to_csv(OUT_PATH, index=False)
print(f"✓ Built commission file with {len(dfc):,} rows → {OUT_PATH}")


✓ Built commission file with 3,071 rows → commission.csv


## Claims

In [26]:
# build_claims_from_policy.py
import pandas as pd
import numpy as np
import random
from datetime import timedelta
from faker import Faker

random.seed(20251008)
np.random.seed(20251008)
fake = Faker("en_US")

POLICY_PATH = "policy.csv"
OUT_PATH    = "claims.csv"

# -------------------- helpers --------------------
def clipf(x, lo, hi):
    return max(lo, min(hi, x))

def days_between(a, b):
    if pd.isna(a) or pd.isna(b): return 0
    return max(0, (b - a).days)

def month_end(d: pd.Timestamp) -> pd.Timestamp:
    if pd.isna(d): return pd.NaT
    y, m = d.year, d.month
    first_next = pd.Timestamp(year=y + (m==12), month=(1 if m==12 else m+1), day=1)
    return first_next - pd.Timedelta(days=1)

def pick(cats, probs):
    return np.random.choice(cats, p=np.array(probs)/np.sum(probs))

# Severity (ultimate incurred) by LOB (lognormal-ish medians)
def draw_ultimate(lob: str):
    # medians are rough order-of-magnitude; spread controls tail
    specs = {
        "Personal Auto": (2500, 0.9, 200, 50000),
        "Homeowners":    (6000, 1.0, 500, 150000),
        "BOP":           (12000, 1.1, 1000, 300000),
        "WC":            (15000, 1.2, 1000, 500000),
        "Umbrella":      (25000, 1.1, 5000, 1000000),
        "Package":       (18000, 1.1, 1500, 400000),
    }
    median, spread, lo, hi = specs.get(lob, (8000, 1.0, 500, 250000))
    mu = np.log(median)
    amt = float(np.random.lognormal(mean=mu, sigma=spread))
    return round(clipf(amt, lo, hi), 2)

# Cause of loss by LOB
CAUSES = {
    "Personal Auto": (["Collision","Comprehensive Theft","Glass","Weather","Liability BI/PD"], [0.55,0.10,0.10,0.10,0.15]),
    "Homeowners":    (["Wind/Hail","Water (non-flood)","Fire","Theft/Vandalism","Liability"], [0.40,0.30,0.15,0.08,0.07]),
    "BOP":           (["Wind/Hail","Water","Fire","Burglary","Liability"], [0.30,0.25,0.20,0.10,0.15]),
    "WC":            (["Strain/Sprain","Slip/Fall","Cut/Puncture","Vehicle","Other"], [0.35,0.25,0.15,0.10,0.15]),
    "Umbrella":      (["Excess Auto","Excess GL","Excess HO","Catastrophic"], [0.50,0.30,0.15,0.05]),
    "Package":       (["Property Wind/Hail","Property Fire","GL Prem/Ops","Theft","Water"], [0.30,0.20,0.25,0.10,0.15]),
}

def cause_for(lob):
    cats, probs = CAUSES.get(lob, (["Other"], [1.0]))
    return pick(cats, probs)

# Frequency (expected claims per policy/term) by LOB

LAMBDA = {
    "Personal Auto": 0.18,  # was 0.10
    "Homeowners":    0.10,  # was 0.06
    "BOP":           0.08,  # was 0.05
    "WC":            0.07,  # was 0.04
    "Umbrella":      0.02,  # was 0.01
    "Package":       0.07,  # was 0.04
}


def poisson(lmbda):
    # allow zeros; cap at, say, 4 to avoid absurd outliers
    return int(min(np.random.poisson(lmbda), 4))

def split_paid_reserve(ultimate, age_days, status):
    """Return (paid_loss, paid_exp, reserve_loss, reserve_exp) with realistic proportions."""
    # ALAE ~ 10–25% of indemnity on average (very rough), with tail
    alae_ratio = clipf(np.random.normal(0.18, 0.07), 0.05, 0.40)
    ultimate_loss = ultimate / (1 + alae_ratio)
    ultimate_exp  = ultimate - ultimate_loss

    # How much of ultimate is paid vs reserved depends on age & status
    if status == "Closed":
        paid_loss = ultimate_loss
        paid_exp  = ultimate_exp
        res_loss = 0.0
        res_exp  = 0.0
    else:
        # Open/Reopened: pay some fraction to-date
        # older claims → higher paid fraction
        paid_frac = clipf(np.random.normal(0.35 + 0.003*age_days, 0.12), 0.05, 0.95)
        paid_loss = ultimate_loss * paid_frac
        paid_exp  = ultimate_exp  * clipf(np.random.normal(0.60, 0.15), 0.20, 0.95)
        res_loss = ultimate_loss - paid_loss
        res_exp  = max(0.0, ultimate_exp - paid_exp)

    return round(paid_loss,2), round(paid_exp,2), round(res_loss,2), round(res_exp,2)

def unique_claim_numbers(loss_dates, start_seq=1):
    """Yield YYYY-###### numbers in order of loss date; guarantees uniqueness."""
    seq = start_seq
    out = []
    for d in loss_dates:
        year = pd.Timestamp(d).year if not pd.isna(d) else 2025
        out.append(f"{year}-{seq:06d}")
        seq += 1
    return out

def adjuster_contact():
    name = f"{fake.first_name()[0]}. {fake.last_name()}"
    phone = fake.phone_number()
    return f"{name}, {phone}"

# -------------------- load policy --------------------
dfp = pd.read_csv(POLICY_PATH)
must = ["PolicyNumber","CarrierCode","AgentCode","LineOfBusiness","PolicyEffectiveDate","PolicyExpirationDate","RiskLocationAddress","State"]
for c in must:
    if c not in dfp.columns:
        raise ValueError(f"Missing required policy column: {c}")

for c in ["PolicyEffectiveDate","PolicyExpirationDate"]:
    dfp[c] = pd.to_datetime(dfp[c], errors="coerce")

# -------------------- build claims --------------------
rows = []
for i, r in dfp.iterrows():
    lob   = r["LineOfBusiness"]
    lmbda = LAMBDA.get(lob, 0.03)
    n_claims = poisson(lmbda)
    if n_claims == 0:
        continue

    eff = r["PolicyEffectiveDate"]
    exp = r["PolicyExpirationDate"]
    pol = r["PolicyNumber"]

    for _ in range(n_claims):
        # Loss during the term (uniform); report delay 0–7 days (skew to 1–2)
        if pd.isna(eff) or pd.isna(exp) or eff >= exp:
            continue
        loss = eff + pd.Timedelta(days=int(np.random.uniform(0, (exp - eff).days)))
        report_delay = int(clipf(np.random.gamma(shape=1.5, scale=1.0), 0, 21))  # mean ~1–2 days, cap 3 weeks
        report = loss + pd.Timedelta(days=report_delay)

        # Status logic: older claims are more likely closed; small chance of reopen
        age_days = days_between(loss, min(report, exp))
        # Base close prob grows with time since loss
        close_prob = clipf(0.10 + 0.0025 * days_between(loss, exp), 0.10, 0.85)
        is_closed  = np.random.uniform(0,1) < close_prob
        reopened   = False
        close_date = pd.NaT
        reopen_date= pd.NaT

        if is_closed:
            # close between report and min(exp, report + 180)
            close_span = max(7, int(np.random.normal(60, 30)))
            close_date = min(exp, report + timedelta(days=abs(close_span)))
            # small reopen chance
            if np.random.uniform(0,1) < 0.06:
                reopened = True
                reopen_date = min(exp, close_date + timedelta(days=int(np.random.uniform(5, 90))))

        status = "Reopened" if reopened else ("Closed" if is_closed else "Open")

        # Ultimate incurred severity by LOB
        ultimate = draw_ultimate(lob)

        # Paid/Reserve split
        age_for_paid = days_between(loss, (reopen_date if reopened else (close_date if is_closed else report)))
        paid_loss, paid_exp, res_loss, res_exp = split_paid_reserve(ultimate, age_for_paid, status)

        # Loss location
        def address_in_state(state_abbr: str) -> str:
            # build a plausible in-state address using Faker parts + fixed state
            street = fake.street_address()
            city = fake.city()
            zipc = fake.postcode()
            return f"{street}, {city}, {state_abbr} {zipc}"

        # per-LOB probability that LossLocation == RiskLocationAddress
        LOSS_SAME_ADDR_PROB = {
            "Homeowners":    0.85,  # most HO losses at the insured location
            "Personal Auto": 0.30,  # mostly away from home/garaging location
            "BOP":           0.40,  # many losses at other premises/away
            "WC":            0.50,  # could be job site or employer location
            "Umbrella":      0.40,  # follows underlying, often away
            "Package":       0.45,
        }

        # Loss location: same state, often different address for non-HO
        policy_state = r["State"]
        risk_addr = r["RiskLocationAddress"]
        same_prob = LOSS_SAME_ADDR_PROB.get(lob, 0.50)
        use_same = (random.random() < same_prob)

        if use_same:
            loss_addr = risk_addr
        else:
            # ensure a different string while keeping the same state
            tries = 0
            candidate = address_in_state(policy_state)
            while candidate == risk_addr and tries < 3:
                candidate = address_in_state(policy_state)
                tries += 1
            loss_addr = candidate


        # Adjuster
        adj = adjuster_contact()

        rows.append({
            "CarrierCode": r["CarrierCode"],
            "AgentCode": r["AgentCode"],
            "PolicyNumber": pol,
            "ClaimNumber": None,                    # fill later
            "LossDate": loss.date(),
            "ReportDate": report.date(),
            "ClaimStatus": status,
            "CauseOfLoss": cause_for(lob),
            "LossLocation": loss_addr,
            "PaidLossAmount": round(paid_loss, 2),
            "PaidExpenseAmount": round(paid_exp, 2),
            "ReserveLossAmount": round(0.0 if status=="Closed" else res_loss, 2),
            "ReserveExpenseAmount": round(0.0 if status=="Closed" else res_exp, 2),
            "CloseDate": (None if pd.isna(close_date) else close_date.date()),
            "ReopenDate": (None if pd.isna(reopen_date) else reopen_date.date()),
            "AdjusterContact": adj,
        })

# Assemble DF
dfc = pd.DataFrame(rows, columns=[
    "CarrierCode","AgentCode","PolicyNumber","ClaimNumber","LossDate","ReportDate",
    "ClaimStatus","CauseOfLoss","LossLocation",
    "PaidLossAmount","PaidExpenseAmount","ReserveLossAmount","ReserveExpenseAmount",
    "CloseDate","ReopenDate","AdjusterContact"
])

# Unique claim numbers (YYYY-###### sorted by loss date)
if not dfc.empty:
    dfc = dfc.sort_values("LossDate").reset_index(drop=True)
    dfc["ClaimNumber"] = unique_claim_numbers(dfc["LossDate"], start_seq=1)

# Final polish: types & realism
money_cols = ["PaidLossAmount","PaidExpenseAmount","ReserveLossAmount","ReserveExpenseAmount"]
for c in money_cols:
    if c in dfc.columns:
        dfc[c] = dfc[c].astype(float).round(2)

# Optional sanity: closed → zero reserves
if not dfc.empty:
    closed_mask = dfc["ClaimStatus"].eq("Closed")
    dfc.loc[closed_mask, ["ReserveLossAmount","ReserveExpenseAmount"]] = 0.00

# Write
dfc.to_csv(OUT_PATH, index=False)
print(f"✓ Built claims file with {len(dfc):,} rows → {OUT_PATH}")


✓ Built claims file with 322 rows → claims.csv


## KPIs

In [ ]:
# build_kpi.py
# Create KPI table from policy.csv, commission.csv, claims.csv
# Definitions:
#   RetentionRate = Renewals / EligibleForRenewal
#   LossRatio = IncurredLoss / EarnedPremium
#   CommissionRatio = CommissionAmount / WrittenPremium (commission lines)
#   RewrittenPolicyRatio = Rewrite transactions / New transactions
#
# Notes:
# - Aggregation grain is configurable (portfolio / by carrier / by carrier+agent)
# - Period window default: 2025-04-01 → 2026-03-31 (inclusive)
# - Uses an expiration BUFFER of 1 month to avoid empty renewal eligibles
# - Ratios rounded to 2 decimals; counts/amounts kept numeric

import pandas as pd
import numpy as np
from pandas.tseries.offsets import DateOffset

# ------------------------- Config -------------------------
POLICY_PATH     = "policy.csv"
COMMISSION_PATH = "commission.csv"
CLAIMS_PATH     = "claims.csv"
OUT_PATH        = "kpi.csv"

# Aggregation grain: [] (one row portfolio), ["CarrierCode"], or ["CarrierCode","AgentCode"]
AGGREGATION = ["AgentCode"]

# Measurement window (inclusive)
PERIOD_START = pd.Timestamp(2025, 4, 1)
PERIOD_END   = pd.Timestamp(2026, 3, 31)

# Renewal eligibility: allow expirations up to 1 extra month (helps synthetic 12m terms)
EXPIRATION_BUFFER = DateOffset(months=1)

# Guards
MIN_COMMISSION_BASIS = 100.0   # require at least $100 absolute WP in cell for CommissionRatio
MIN_EARNED_FOR_LR    = 0.0     # set >0 if you want to blank LR on tiny earned

INCLUDE_OVERALL_ROW  = True    # add an "ALL" rollup row at the end
# ----------------------------------------------------------

# ------------------------- Load ---------------------------
pol = pd.read_csv(POLICY_PATH)
com = pd.read_csv(COMMISSION_PATH)
clm = pd.read_csv(CLAIMS_PATH)

# Parse dates
for c in ["PolicyEffectiveDate","PolicyExpirationDate","TransactionEffectiveDate"]:
    pol[c] = pd.to_datetime(pol[c], errors="coerce")
for c in ["StatementDate","DueDate"]:
    if c in com.columns:
        com[c] = pd.to_datetime(com[c], errors="coerce")
for c in ["LossDate","ReportDate","CloseDate","ReopenDate"]:
    if c in clm.columns:
        clm[c] = pd.to_datetime(clm[c], errors="coerce")

# --------------------- Helper functions -------------------
def earned_within_period(eff, exp, term_prem, p_start, p_end):
    """
    Daily pro-rata earned for the overlap of [eff, exp] with [p_start, p_end], inclusive of endpoints.
    """
    if pd.isna(eff) or pd.isna(exp) or exp <= eff or pd.isna(term_prem):
        return 0.0
    start = max(eff, p_start)
    end   = min(exp, p_end)
    if end < start:
        return 0.0
    term_days = max(1, (exp - eff).days)
    overlap_days = (end - start).days + 1
    ratio = np.clip(overlap_days / term_days, 0.0, 1.0)
    return float(term_prem) * ratio

def safe_ratio(numer, denom, min_abs=0.0):
    if denom is None or np.isnan(denom) or abs(denom) < min_abs:
        return np.nan
    return numer / denom

# ------------------- Policy-derived pieces ----------------
# Earned in period for any policy overlapping the window
inforce = pol[(pol["PolicyExpirationDate"] >= PERIOD_START) & (pol["PolicyEffectiveDate"] <= PERIOD_END)].copy()
inforce["EarnedInPeriod"] = inforce.apply(
    lambda r: earned_within_period(r["PolicyEffectiveDate"], r["PolicyExpirationDate"], r["TermPremium"], PERIOD_START, PERIOD_END),
    axis=1
)
earn = inforce.groupby(AGGREGATION, dropna=False, as_index=False).agg(
    EarnedPremium=("EarnedInPeriod","sum"),
    PoliciesActive=("PolicyNumber","count")
)

# Retention (eligibles by expiration with 1-month buffer; renewals by effective date in period)
elig = pol[(pol["PolicyExpirationDate"] >= PERIOD_START) &
           (pol["PolicyExpirationDate"] <= PERIOD_END + EXPIRATION_BUFFER)].copy()
elig["EligibleForRenewal"] = 1

ren = pol[(pol["PolicyEffectiveDate"] >= PERIOD_START) &
          (pol["PolicyEffectiveDate"] <= PERIOD_END) &
          (pol["TransactionType"] == "Renewal")].copy()
ren["Renewals"] = 1

ret_elig = elig.groupby(AGGREGATION, dropna=False, as_index=False).agg(EligibleForRenewal=("EligibleForRenewal","sum"))
ret_ren  = ren.groupby(AGGREGATION,   dropna=False, as_index=False).agg(Renewals=("Renewals","sum"))
ret = ret_elig.merge(ret_ren, on=AGGREGATION, how="left")
ret["Renewals"] = ret["Renewals"].fillna(0)
ret["RetentionRate"] = np.where(ret["EligibleForRenewal"] > 0,
                                ret["Renewals"] / ret["EligibleForRenewal"],
                                np.nan)

# Rewrites vs New (transaction effective date in period)
tx = pol[(pol["TransactionEffectiveDate"] >= PERIOD_START) &
         (pol["TransactionEffectiveDate"] <= PERIOD_END)].copy()
tx["NewCount"] = (tx["TransactionType"] == "New").astype(int)
tx["RewriteCount"] = (tx["TransactionType"] == "Rewrite").astype(int)
rw = tx.groupby(AGGREGATION, dropna=False, as_index=False).agg(
    NewCount=("NewCount","sum"),
    RewriteCount=("RewriteCount","sum")
)
rw["RewrittenPolicyRatio"] = np.where(rw["NewCount"] > 0,
                                      rw["RewriteCount"] / rw["NewCount"],
                                      np.nan)

# ----------------- Commission-derived pieces --------------
# Use commission lines with StatementDate in period
com_p = com[(com["StatementDate"] >= PERIOD_START) & (com["StatementDate"] <= PERIOD_END)].copy()
comg = com_p.groupby(AGGREGATION, dropna=False, as_index=False).agg(
    CommissionWrittenPremium=("WrittenPremium","sum"),
    CommissionAmount=("CommissionAmount","sum")
)
# Guard tiny denominators
basis_adj = comg["CommissionWrittenPremium"].where(comg["CommissionWrittenPremium"].abs() >= MIN_COMMISSION_BASIS, np.nan)
comg["CommissionRatio"] = comg["CommissionAmount"] / basis_adj

# ------------------- Claims-derived pieces ----------------
clm_p = clm[(clm["LossDate"] >= PERIOD_START) & (clm["LossDate"] <= PERIOD_END)].copy()
clm_p["IncurredLoss"] = clm_p["PaidLossAmount"].astype(float).fillna(0.0) + \
                        clm_p["ReserveLossAmount"].astype(float).fillna(0.0)
loss = clm_p.groupby(AGGREGATION, dropna=False, as_index=False).agg(IncurredLoss=("IncurredLoss","sum"))

# ---------------------- Assemble KPI ----------------------
kpi = earn.merge(ret, on=AGGREGATION, how="outer") \
          .merge(comg, on=AGGREGATION, how="outer") \
          .merge(loss, on=AGGREGATION, how="outer") \
          .merge(rw, on=AGGREGATION, how="outer")

# Fill numeric nulls for totals; keep ratios NaN if denom missing
for c in ["EarnedPremium","PoliciesActive","EligibleForRenewal","Renewals",
          "CommissionWrittenPremium","CommissionAmount","IncurredLoss",
          "NewCount","RewriteCount"]:
    if c in kpi.columns:
        kpi[c] = kpi[c].fillna(0.0)

# Loss ratio with optional earned guard
kpi["LossRatio"] = np.where(kpi["EarnedPremium"] > MIN_EARNED_FOR_LR,
                            kpi["IncurredLoss"] / kpi["EarnedPremium"],
                            np.nan)

# Period labels
kpi["PeriodStart"] = PERIOD_START.date()
kpi["PeriodEnd"]   = PERIOD_END.date()

# Order columns
base_cols = AGGREGATION + ["PeriodStart","PeriodEnd",
    "RetentionRate","LossRatio","CommissionRatio","RewrittenPolicyRatio",
    "EarnedPremium","IncurredLoss","CommissionAmount","CommissionWrittenPremium",
    "EligibleForRenewal","Renewals","PoliciesActive","NewCount","RewriteCount"
]
kpi = kpi[base_cols]

# Optional ALL row (rollup across aggregation)
if INCLUDE_OVERALL_ROW:
    overall = kpi.copy()
    # mark all dims as "ALL"
    if AGGREGATION:
        for dim in AGGREGATION:
            overall[dim] = "ALL"
    # sum numeric, then recompute ratios
    group_keys = (AGGREGATION if AGGREGATION else []) + ["PeriodStart","PeriodEnd"]
    overall = overall.groupby(group_keys, as_index=False).sum(numeric_only=True)

    # recompute portfolio ratios with guards
    overall["RetentionRate"] = np.where(overall["EligibleForRenewal"]>0,
                                        overall["Renewals"]/overall["EligibleForRenewal"], np.nan)
    overall["CommissionRatio"] = np.where(overall["CommissionWrittenPremium"].abs() >= MIN_COMMISSION_BASIS,
                                          overall["CommissionAmount"]/overall["CommissionWrittenPremium"], np.nan)
    overall["LossRatio"] = np.where(overall["EarnedPremium"] > MIN_EARNED_FOR_LR,
                                    overall["IncurredLoss"]/overall["EarnedPremium"], np.nan)
    overall["RewrittenPolicyRatio"] = np.where(overall["NewCount"]>0,
                                               overall["RewriteCount"]/overall["NewCount"], np.nan)
    # Append
    kpi = pd.concat([kpi, overall], ignore_index=True)

# Round ratios to 2 decimals
for ratio_col in ["RetentionRate","LossRatio","CommissionRatio","RewrittenPolicyRatio"]:
    if ratio_col in kpi.columns:
        kpi[ratio_col] = kpi[ratio_col].round(2)

# Write
kpi.to_csv(OUT_PATH, index=False)
print(f"✓ KPI rows: {len(kpi)} — aggregation = {AGGREGATION or 'PORTFOLIO'} — saved to {OUT_PATH}")


✓ KPI rows: 301 — aggregation = ['AgentCode'] — saved to kpi.csv


## Submissions

In [5]:
"""
make_submissions_from_policy.py

Creates mock submission data that is CONSISTENT with your policy file:
- submissions.csv     (SubmissionId, AccountId, AgentCode, CarrierId, LOB, EffDt, Channel, Status)
- submission_items.csv(SubmissionItemId, SubmissionId, ClassCode, NAICS, Exposure, AssetDetails)
- kpi_with_submissions.csv (only if kpi.csv exists with AgentCode, PeriodStart, PeriodEnd)
    -> adds SubmissionQuality = Bound / Submitted per AgentCode within each KPI row's period.

Usage:
  python make_submissions_from_policy.py
"""

import numpy as np
import pandas as pd
from pathlib import Path

# --------------------------
# Config (tweak if needed)
# --------------------------
INPUT_DIR  = Path(".")
OUTPUT_DIR = Path(".")
MAX_SEED_POLICIES = 20000            # cap if policy is huge (for speed)
SUBS_MIN, SUBS_MAX = 2, 4            # total submissions per policy (incl. the bound one)
QUOTE_PROB = 0.65                    # competitor quote probability
RANDOM_SEED = 123

# --------------------------
# Helpers
# --------------------------
def pick_first(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def parse_dates(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")
    return df

def norm_str(x):
    return "" if pd.isna(x) else str(x).strip()

# --------------------------
# Main
# --------------------------
def main():
    np.random.seed(RANDOM_SEED)

    policy_path = INPUT_DIR / "policy.csv"
    if not policy_path.exists():
        raise FileNotFoundError(f"policy.csv not found at {policy_path.resolve()}")

    policy = pd.read_csv(policy_path)

    # Detect columns from your real file
    agent_col   = pick_first(policy, ["AgentCode","AgencyCode","Agency","ProducerCode"])
    carrier_col = pick_first(policy, [
        "CarrierCode", "Carrier", "CarrierId", "Company", "Insurer", "Market"  # <- includes CarrierCode
    ])
    lob_col     = pick_first(policy, ["LOB","LineOfBusiness","Line","Product","LOBCode"])
    eff_col     = pick_first(policy, ["PolicyEffectiveDate","EffectiveDate","InceptionDate"])
    exp_col     = pick_first(policy, ["PolicyExpirationDate","ExpirationDate","ExpiryDate"])
    acct_col    = pick_first(policy, ["AccountId","InsuredId","CustomerId","Account","Insured","InsuredName","NamedInsured"])
    polnum_col  = pick_first(policy, ["PolicyNumber","POLICYNUMBER","policy_number"])

    if agent_col is None:
        raise RuntimeError("No Agent column found (tried AgentCode/AgencyCode/Agency/ProducerCode).")
    if eff_col is None:
        raise RuntimeError("No effective date column found (e.g., PolicyEffectiveDate).")
    if carrier_col is None:
        raise RuntimeError("No carrier column found (looked for CarrierCode/Carrier/CarrierId/Company/Insurer/Market).")

    policy = parse_dates(policy, [eff_col, exp_col] if exp_col else [eff_col])

    # Universes from YOUR data
    carriers = sorted(pd.Series(policy[carrier_col].dropna().astype(str).unique()).tolist())
    if not carriers:
        raise RuntimeError("Carrier universe from policy.csv is empty. Ensure that column has values (e.g., TRAV, ALLS, PRGS, AMFA).")

    lobs = sorted(pd.Series(policy[lob_col].dropna().astype(str).unique()).tolist()) if lob_col else []
    if not lobs:
        lobs = ["Unknown"]

    # Ensure an AccountId: use AccountId if present; else derive stable from PolicyNumber or row index
    if acct_col is None:
        if polnum_col is not None:
            acct_col = "__AccountId__"
            policy[acct_col] = policy[polnum_col].astype(str).map(lambda x: f"A-{abs(hash(x))%1000000:06d}")
        else:
            acct_col = "__AccountId__"
            policy[acct_col] = [f"A-{i:06d}" for i in range(len(policy))]

    # Seed policy rows with valid effective dates
    seed = policy.dropna(subset=[eff_col]).copy()
    if MAX_SEED_POLICIES and len(seed) > MAX_SEED_POLICIES:
        seed = seed.sample(MAX_SEED_POLICIES, random_state=RANDOM_SEED).sort_values(eff_col)

    # Build submissions
    channels = ["IVANS","Portal","Email"]
    chan_p   = [0.45, 0.45, 0.10]

    sub_rows = []

    def submission_date_before(eff_dt):
        # set submission date 7–60 days before policy effective
        return eff_dt - pd.Timedelta(days=int(np.random.randint(7, 61)))

    for _, r in seed.iterrows():
        eff_dt = r[eff_col]
        if pd.isna(eff_dt):
            continue

        agent   = norm_str(r[agent_col])
        account = norm_str(r[acct_col])
        lob     = norm_str(r[lob_col]) if lob_col else "Unknown"
        bound_carrier = norm_str(r[carrier_col])

        # Always include the bound submission (the policy’s carrier)
        sub_rows.append({
            "SubmissionId": f"S-{len(sub_rows)+1:06d}",
            "AccountId": account,
            "AgentCode": agent,
            "CarrierId": bound_carrier,    # your real code, e.g., TRAV, ALLS, PRGS, AMFA
            "LOB": lob,
            "EffDt": submission_date_before(eff_dt),
            "Channel": np.random.choice(channels, p=chan_p),
            "Status": "Bound"
        })

        # Plus 1–3 competitor submissions using YOUR carrier codes
        n_total = int(np.random.randint(SUBS_MIN, SUBS_MAX + 1))
        n_comp  = max(0, n_total - 1)
        competitor_pool = [c for c in carriers if c != bound_carrier] or carriers

        for _ in range(n_comp):
            car = np.random.choice(competitor_pool)
            quoted = (np.random.rand() < QUOTE_PROB)
            status = "Quoted" if quoted else "Declined"
            sub_rows.append({
                "SubmissionId": f"S-{len(sub_rows)+1:06d}",
                "AccountId": account,
                "AgentCode": agent,
                "CarrierId": car,          # also your real codes
                "LOB": lob,
                "EffDt": submission_date_before(eff_dt),
                "Channel": np.random.choice(channels, p=chan_p),
                "Status": status
            })

    submissions = pd.DataFrame(sub_rows)

    # Submission items (detail rows; not required for hit ratio)
    item_rows = []
    CLASS_CODES = ["8810","9082","7228","5403","5606","3724","3632","8397"]
    NAICS_CODES = ["541612","238320","238220","541511","722513","484110","236115","561720"]

    item_id = 1
    for sid in submissions["SubmissionId"]:
        n_items = int(np.random.randint(1, 4))  # 1–3 items
        for _ in range(n_items):
            item_rows.append({
                "SubmissionItemId": f"SI-{item_id:07d}",
                "SubmissionId": sid,
                "ClassCode": np.random.choice(CLASS_CODES),
                "NAICS": np.random.choice(NAICS_CODES),
                "Exposure": round(float(np.random.lognormal(mean=10.4, sigma=0.7)), 2),
                "AssetDetails": np.random.choice(["Truck","Office","Warehouse","Crew","Equipment","Storefront"])
            })
            item_id += 1

    submission_items = pd.DataFrame(item_rows)

    # Write outputs
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    submissions.to_csv(OUTPUT_DIR / "submissions.csv", index=False)
    submission_items.to_csv(OUTPUT_DIR / "submission_items.csv", index=False)
    print(f"Wrote {OUTPUT_DIR / 'submissions.csv'} ({len(submissions)} rows)")
    print(f"Wrote {OUTPUT_DIR / 'submission_items.csv'} ({len(submission_items)} rows)")

    # ----- Add SubmissionQuality to KPI (if kpi.csv exists with periods) -----
    kpi_path = INPUT_DIR / "kpi.csv"
    if kpi_path.exists():
        kpi = pd.read_csv(kpi_path)
        has_periods = all(c in kpi.columns for c in ["AgentCode","PeriodStart","PeriodEnd"])
        if has_periods:
            for c in ["PeriodStart","PeriodEnd"]:
                kpi[c] = pd.to_datetime(kpi[c], errors="coerce")

            # normalize status → bound & quoted flags
            st_norm = submissions["Status"].astype(str).str.strip().str.casefold()
            submissions["__is_bound__"]  = st_norm.eq("bound").astype(int)
            submissions["__is_quoted__"] = st_norm.isin(["quoted","bound"]).astype(int)

            def submission_quality(agent, start, end):
                if pd.isna(start) or pd.isna(end):
                    return np.nan
                block = submissions[
                    (submissions["AgentCode"].astype(str) == str(agent)) &
                    (submissions["EffDt"] >= start) &
                    (submissions["EffDt"] <= end)
                ]
                submitted = block["SubmissionId"].nunique()
                if submitted == 0:
                    return np.nan
                bound = int(block["__is_bound__"].sum())
                # Use Bound / Submitted (switch to Bound / Quoted if you prefer)
                return round(bound / submitted, 4)

            kpi["SubmissionQuality"] = kpi.apply(
                lambda r: submission_quality(r["AgentCode"], r["PeriodStart"], r["PeriodEnd"]), axis=1
            )
            kpi.to_csv(OUTPUT_DIR / "kpi_with_submissions.csv", index=False)
            print(f"Wrote {OUTPUT_DIR / 'kpi_with_submissions.csv'} with SubmissionQuality.")
        else:
            print("kpi.csv found, but missing AgentCode/PeriodStart/PeriodEnd — skipped SubmissionQuality merge.")

if __name__ == "__main__":
    main()


Wrote submissions.csv (9093 rows)
Wrote submission_items.csv (18282 rows)
Wrote kpi_with_submissions.csv with SubmissionQuality.


In [6]:
"""
merge_submission_quality.py

Compute SubmissionQuality (Bound / Submitted) for each KPI row and write kpi_with_submissions.csv.
Adds debug columns so you can see counts and the actual window used.

Assumes:
  - kpi.csv has columns: AgentCode, PeriodStart, PeriodEnd
  - submissions.csv has columns: SubmissionId, AgentCode, EffDt, Status

Run:
  python merge_submission_quality.py

If your submissions were generated with make_submissions_from_policy.py, EffDt is 7–60 days
before policy effective. Use WINDOW_START_OFFSET_DAYS=60 (or bigger) so those submissions
count toward the period that starts at PeriodStart.
"""

import pandas as pd
import numpy as np
from pathlib import Path

# --- Config
INPUT_DIR = Path(".")
OUTPUT_DIR = Path(".")
WINDOW_START_OFFSET_DAYS = 60    # include subs up to 60d BEFORE PeriodStart
DENOMINATOR = "submitted"        # "submitted" or "quoted"  (Bound / Submitted) vs (Bound / Quoted)

# --- Load
kpi_path = INPUT_DIR / "kpi.csv"
subs_path = INPUT_DIR / "submissions.csv"

if not kpi_path.exists():
    raise FileNotFoundError(f"Missing {kpi_path}")
if not subs_path.exists():
    raise FileNotFoundError(f"Missing {subs_path}")

kpi = pd.read_csv(kpi_path)
subs = pd.read_csv(subs_path)

# --- Parse dates (be liberal with formats)
for c in ["PeriodStart","PeriodEnd","EffDt"]:
    if c in kpi.columns:
        kpi[c] = pd.to_datetime(kpi[c], errors="coerce", infer_datetime_format=True)
    if c in subs.columns:
        subs[c] = pd.to_datetime(subs[c], errors="coerce", infer_datetime_format=True)

# --- Normalize required cols
if "AgentCode" not in kpi.columns:
    raise RuntimeError("kpi.csv must include AgentCode")

needed_sub_cols = {"SubmissionId","AgentCode","EffDt","Status"}
missing = needed_sub_cols - set(subs.columns)
if missing:
    raise RuntimeError(f"submissions.csv missing columns: {missing}")

# Normalize types (avoid str vs int mismatch)
kpi["AgentCode"]  = kpi["AgentCode"].astype(str).str.strip()
subs["AgentCode"] = subs["AgentCode"].astype(str).str.strip()

# Status flags
status_norm = subs["Status"].astype(str).str.strip().str.casefold()
subs["__is_bound__"]  = status_norm.eq("bound").astype(int)
subs["__is_quoted__"] = status_norm.isin(["quoted","bound"]).astype(int)

# --- Compute per-row SubmissionQuality with an offset window
def compute_row(row):
    ag  = str(row["AgentCode"])
    ps  = row.get("PeriodStart")
    pe  = row.get("PeriodEnd")
    if pd.isna(ps) or pd.isna(pe):
        return pd.Series({"SubmissionQuality": np.nan, "Submitted": np.nan, "Quoted": np.nan,
                          "Bound": np.nan, "WindowStartUsed": pd.NaT, "WindowEndUsed": pd.NaT})

    start = ps - pd.Timedelta(days=WINDOW_START_OFFSET_DAYS)
    end   = pe

    block = subs[(subs["AgentCode"] == ag) &
                 (subs["EffDt"] >= start) &
                 (subs["EffDt"] <= end)]

    submitted = block["SubmissionId"].nunique()
    quoted    = int(block["__is_quoted__"].sum())
    bound     = int(block["__is_bound__"].sum())

    if DENOMINATOR == "quoted":
        denom = quoted
    else:
        denom = submitted

    quality = (bound / denom) if denom and denom > 0 else np.nan
    return pd.Series({
        "SubmissionQuality": round(quality, 4) if pd.notna(quality) else np.nan,
        "Submitted": submitted,
        "Quoted": quoted,
        "Bound": bound,
        "WindowStartUsed": start,
        "WindowEndUsed": end
    })

out = kpi.copy()
calc = out.apply(compute_row, axis=1)
out = pd.concat([out, calc], axis=1)

# --- Save
out_path = OUTPUT_DIR / "kpi_with_submissions.csv"
out.to_csv(out_path, index=False)
print(f"Wrote {out_path} with SubmissionQuality and debug columns.")

# --- Quick diagnostics to STDOUT
null_rows = out["SubmissionQuality"].isna().sum()
total_rows = len(out)
print(f"Rows with NULL SubmissionQuality: {null_rows}/{total_rows}")

# Show agents with zero submitted in their windows (top 10)
zero_mask = (out["Submitted"].fillna(0) == 0)
if zero_mask.any():
    print("\nAgents with 0 Submitted within (PeriodStart-Offset, PeriodEnd):")
    print(out.loc[zero_mask, ["AgentCode","PeriodStart","PeriodEnd","WindowStartUsed","WindowEndUsed"]].head(10).to_string(index=False))


C:\Users\SujaySunilNagvekar\AppData\Local\Temp\ipykernel_32704\2760930608.py:44: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  kpi[c] = pd.to_datetime(kpi[c], errors="coerce", infer_datetime_format=True)
C:\Users\SujaySunilNagvekar\AppData\Local\Temp\ipykernel_32704\2760930608.py:44: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  kpi[c] = pd.to_datetime(kpi[c], errors="coerce", infer_datetime_format=True)
C:\Users\SujaySunilNagvekar\AppData\Local\Temp\ipykernel_32704\2760930608.py:46: UserWarning: The argument 'infer_datetime_format' is deprecated and will 

Wrote kpi_with_submissions.csv with SubmissionQuality and debug columns.
Rows with NULL SubmissionQuality: 1/301

Agents with 0 Submitted within (PeriodStart-Offset, PeriodEnd):
AgentCode PeriodStart  PeriodEnd WindowStartUsed WindowEndUsed
      ALL  2025-04-01 2026-03-31      2025-01-31    2026-03-31


In [18]:
from scipy.stats import norm  # add this import

def _rank_to_truncnorm(s: pd.Series, lo: float, hi: float, mu: float | None = None, sigma: float | None = None):
    """
    Rank-preserving map: values -> truncated Normal on [lo, hi].
    - Compute percentile ranks of s
    - Map percentiles through the inverse CDF of a Normal truncated to [lo, hi]
    - mu defaults to midpoint, sigma defaults to (hi-lo)/6 so ~99.7% is inside
    """
    x = s.astype(float).copy()
    mask = x.notna()
    if mask.sum() == 0:
        return x

    # ranks in (0,1)
    pct = x[mask].rank(method="average", pct=True)

    # choose center & spread
    if mu is None:
        mu = (lo + hi) / 2.0
    if sigma is None or sigma <= 0:
        sigma = (hi - lo) / 6.0  # 3σ to each side ≈ bounds

    # truncate in standard normal space
    a_std = (lo - mu) / sigma
    b_std = (hi - mu) / sigma

    # map through truncated normal inverse CDF
    Fa = norm.cdf(a_std)
    Fb = norm.cdf(b_std)
    p_trunc = Fa + pct * (Fb - Fa)             # rescale percentiles to [Fa, Fb]
    z = norm.ppf(p_trunc).clip(a_std, b_std)   # inverse CDF in std-normal space
    y = mu + sigma * z
    return y.clip(lo, hi)

from scipy.stats import truncnorm  # NEW

def _stable_jitter(n, scale=1e-6, seed=123):
    rng = np.random.default_rng(seed)
    return rng.uniform(-scale, scale, size=n)

def _rank_to_truncnorm_smooth(s: pd.Series, lo: float, hi: float,
                              mu: float | None = None, sigma: float | None = None,
                              jitter_scale: float = 1e-6, seed: int = 123):
    """
    Monotone, rank-preserving map to a truncated Normal on [lo, hi].
    Adds tiny deterministic jitter to break ties cleanly.
    """
    x = pd.to_numeric(s, errors="coerce")
    mask = x.notna()
    if mask.sum() == 0:
        return x

    if mu is None:
        mu = (lo + hi) / 2.0
    if sigma is None or sigma <= 0:
        sigma = (hi - lo) / 6.0  # ~99.7% within bounds

    # tie-break: add tiny, deterministic noise before ranking
    vals = x[mask].to_numpy()
    vals_j = vals + _stable_jitter(len(vals), scale=jitter_scale, seed=seed)

    u = pd.Series(vals_j).rank(method="first", pct=True).to_numpy()  # (0,1]

    a = (lo - mu) / sigma
    b = (hi - mu) / sigma
    y = truncnorm.ppf(u, a=a, b=b, loc=mu, scale=sigma)
    y = np.clip(y, lo, hi)

    out = x.copy()
    out.loc[mask] = y
    return out

from scipy.stats import truncnorm

def _stable_jitter(n, scale=1e-6, seed=123):
    rng = np.random.default_rng(seed)
    return rng.uniform(-scale, scale, size=n)

def _rank_to_truncnorm_smooth(s: pd.Series, lo: float, hi: float,
                              mu: float, sigma: float,
                              jitter_scale: float = 1e-6, seed: int = 456):
    x = pd.to_numeric(s, errors="coerce")
    m = x.notna()
    if m.sum() == 0:
        return x
    v = x[m].to_numpy() + _stable_jitter(m.sum(), jitter_scale, seed)
    u = pd.Series(v).rank(method="first", pct=True).to_numpy()  # (0,1]
    a = (lo - mu) / sigma
    b = (hi - mu) / sigma
    y = truncnorm.ppf(u, a=a, b=b, loc=mu, scale=sigma)
    y = np.clip(y, lo, hi)
    out = x.copy(); out.loc[m] = y
    return out



In [ ]:
# build_kpi_v3.py
import pandas as pd
import numpy as np
from pandas.tseries.offsets import DateOffset
from pathlib import Path

# ========================= Config =========================
POLICY_PATH     = "policy.csv"
COMMISSION_PATH = "commission.csv"
CLAIMS_PATH     = "claims.csv"
SUBMISSIONS_PATH = "submissions.csv"   # optional; used for SubmissionQuality if present

OUT_KPI_PATH          = "kpi_clean.csv"
OUT_TOPSIS_VIEW_PATH  = "kpi_topsis_ready.csv"

# Aggregation grain: [] (one-row portfolio), ["CarrierCode"], ["CarrierCode","AgentCode"], ["AgentCode"], etc.
AGGREGATION = ["AgentCode"]

# Measurement window (inclusive)
PERIOD_START = pd.Timestamp(2025, 4, 1)
PERIOD_END   = pd.Timestamp(2026, 3, 31)

# Renewal eligibility: allow expirations up to 1 extra month (helps synthetic 12m terms)
EXPIRATION_BUFFER = DateOffset(months=1)

# Guards
MIN_COMMISSION_BASIS = 100.0   # require at least $100 absolute WP in cell for CommissionRatio
MIN_EARNED_FOR_LR    = 0.0     # set >0 to blank LR on tiny earned

# Winsorization (heavy-tailed $ metrics) — applied at the final KPI table level (not per group)
WINSORIZE_NUM_COLS = ["EarnedPremium", "IncurredLoss", "CommissionAmount", "CommissionWrittenPremium"]
WINSOR_QUANTS = (0.01, 0.99)   # gentle: cap bottom 1%, top 99%

# Logical bounds for proportions (business rules)
RETENTION_MIN, RETENTION_MAX = 0.20, 1.00
SUBQ_MIN, SUBQ_MAX           = 0.20, 1.00
LR_MIN, LR_MAX               = 0.00, 1.00

# Target bands (post-process, rank-preserving)
LR_TARGET_LO, LR_TARGET_HI         = 0.30, 0.90
RET_TARGET_LO, RET_TARGET_HI       = 0.50, 0.95

INCLUDE_OVERALL_ROW  = True
RANDOM_SEED = 123

# Target band for Expense Ratio (lower is better)
ER_TARGET_LO, ER_TARGET_HI = 0.24, 0.40
ER_MU, ER_SIGMA            = 0.32, 0.03   # center & spread for truncated-normal reshape

ER_LO, ER_HI   = 0.24, 0.40
ER_MU, ER_SIG  = 0.32, 0.03    # center and spread

# Overhead used when we don't have an explicit UW_Expenses column
OPEX_PCT = 0.10   # 10% of EarnedPremium as non-commission overhead (tune if needed)
# ==========================================================


# ========================= Utils ==========================
def earned_within_period(eff, exp, term_prem, p_start, p_end):
    """Daily pro-rata earned for the overlap of [eff, exp] with [p_start, p_end], inclusive.
    Returns 0.0 when dates invalid or no overlap."""
    if pd.isna(eff) or pd.isna(exp) or exp <= eff or pd.isna(term_prem):
        return 0.0
    start = max(eff, p_start)
    end   = min(exp, p_end)
    if end < start:
        return 0.0
    term_days = max(1, (exp - eff).days)
    overlap_days = (end - start).days + 1
    ratio = np.clip(overlap_days / term_days, 0.0, 1.0)
    return float(term_prem) * ratio


def clip_with_flags(s: pd.Series, lo: float, hi: float, prefix: str) -> pd.DataFrame:
    """Return DataFrame with value, raw, clipped, and flags."""
    raw = s.astype(float)
    clipped = raw.clip(lower=lo, upper=hi)
    df = pd.DataFrame({
        f"{prefix}_raw": raw,
        f"{prefix}": clipped,
        f"{prefix}_was_clipped_low": (raw < lo).astype(int),
        f"{prefix}_was_clipped_high": (raw > hi).astype(int),
    })
    return df


def winsorize_columns(df: pd.DataFrame, cols: list, lo_q=0.01, hi_q=0.99) -> pd.DataFrame:
    """Winsorize selected numeric columns in-place; add *_winsor flags."""
    for c in cols:
        if c not in df.columns:
            continue
        s = df[c].astype(float)
        lo = s.quantile(lo_q) if s.notna().any() else None
        hi = s.quantile(hi_q) if s.notna().any() else None
        if lo is None or hi is None:
            continue
        df[f"{c}_was_winsor_low"]  = (s < lo).astype(int)
        df[f"{c}_was_winsor_high"] = (s > hi).astype(int)
        df[c] = s.clip(lower=lo, upper=hi)
    return df


def safe_div(n, d, min_abs=0.0):
    if d is None or (isinstance(d, float) and np.isnan(d)) or abs(d) < min_abs:
        return np.nan
    return n / d


def _quantile_remap(s: pd.Series, lo: float, hi: float):
    """Monotone rank-preserving map: values -> target [lo, hi] by percentile."""
    x = s.astype(float).copy()
    mask = x.notna()
    if mask.sum() == 0:
        return x
    pct = x[mask].rank(method="average", pct=True)
    x[mask] = lo + pct * (hi - lo)
    return x
# ==========================================================


def main():
    np.random.seed(RANDOM_SEED)

    # ------------------------- Load -------------------------
    pol = pd.read_csv(POLICY_PATH)
    com = pd.read_csv(COMMISSION_PATH)
    clm = pd.read_csv(CLAIMS_PATH)

    # Parse dates
    for c in ["PolicyEffectiveDate", "PolicyExpirationDate", "TransactionEffectiveDate"]:
        if c in pol.columns:
            pol[c] = pd.to_datetime(pol[c], errors="coerce")
    for c in ["StatementDate", "DueDate"]:
        if c in com.columns:
            com[c] = pd.to_datetime(com[c], errors="coerce")
    for c in ["LossDate", "ReportDate", "CloseDate", "ReopenDate"]:
        if c in clm.columns:
            clm[c] = pd.to_datetime(clm[c], errors="coerce")

    # ------------------- Policy-derived pieces ----------------
    inforce = pol[
        (pol["PolicyExpirationDate"] >= PERIOD_START) &
        (pol["PolicyEffectiveDate"] <= PERIOD_END)
    ].copy()

    if "TermPremium" not in inforce.columns:
        for alt in ["WrittenPremium", "PolicyPremium", "Term_Premium"]:
            if alt in inforce.columns:
                inforce["TermPremium"] = inforce[alt]
                break
        if "TermPremium" not in inforce.columns:
            inforce["TermPremium"] = 0.0

    inforce["EarnedInPeriod"] = inforce.apply(
        lambda r: earned_within_period(
            r["PolicyEffectiveDate"], r["PolicyExpirationDate"], r["TermPremium"],
            PERIOD_START, PERIOD_END
        ),
        axis=1
    )

    earn = inforce.groupby(AGGREGATION, dropna=False, as_index=False).agg(
        EarnedPremium=("EarnedInPeriod","sum"),
        PoliciesActive=("PolicyNumber","count")
    )

    # Retention
    elig = pol[
        (pol["PolicyExpirationDate"] >= PERIOD_START) &
        (pol["PolicyExpirationDate"] <= PERIOD_END + EXPIRATION_BUFFER)
    ].copy()
    elig["EligibleForRenewal"] = 1

    is_renewal = pol["TransactionType"].astype(str).str.casefold().eq("renewal") if "TransactionType" in pol.columns else pd.Series(False, index=pol.index)
    ren = pol[
        (pol["PolicyEffectiveDate"] >= PERIOD_START) &
        (pol["PolicyEffectiveDate"] <= PERIOD_END) &
        (is_renewal)
    ].copy()
    ren["Renewals"] = 1

    ret_elig = elig.groupby(AGGREGATION, dropna=False, as_index=False).agg(EligibleForRenewal=("EligibleForRenewal","sum"))
    ret_ren  = ren.groupby(AGGREGATION,   dropna=False, as_index=False).agg(Renewals=("Renewals","sum"))

    ret = ret_elig.merge(ret_ren, on=AGGREGATION, how="left")
    ret["Renewals"] = ret["Renewals"].fillna(0.0)
    ret["RetentionRate_raw"] = np.where(ret["EligibleForRenewal"] > 0,
                                        ret["Renewals"] / ret["EligibleForRenewal"],
                                        np.nan)

    # ----------------- Commission-derived pieces --------------
    com_p = com[
        (com["StatementDate"] >= PERIOD_START) & (com["StatementDate"] <= PERIOD_END)
    ].copy()
    comg = com_p.groupby(AGGREGATION, dropna=False, as_index=False).agg(
        CommissionWrittenPremium=("WrittenPremium","sum") if "WrittenPremium" in com_p.columns else ("WrittenPremiumAmount","sum"),
        CommissionAmount=("CommissionAmount","sum")
    )
    basis_adj = comg["CommissionWrittenPremium"].where(comg["CommissionWrittenPremium"].abs() >= MIN_COMMISSION_BASIS, np.nan)
    comg["CommissionRatio_raw"] = comg["CommissionAmount"] / basis_adj

    # Optional expense ratio
    expense_col = None
    for name in ["UnderwritingExpenses", "Expenses", "UW_Expenses"]:
        if name in com_p.columns:
            expense_col = name; break
        if name in pol.columns:
            expense_col = name; break
    exp_df = None
    if expense_col:
        holder = com_p if expense_col in com_p.columns else pol
        exp_df = holder.groupby(AGGREGATION, dropna=False, as_index=False).agg(UW_Expenses=(expense_col, "sum"))

    # ------------------- Claims-derived pieces ----------------
    clm_p = clm[
        (clm["LossDate"] >= PERIOD_START) & (clm["LossDate"] <= PERIOD_END)
    ].copy()
    clm_p["IncurredLoss"] = clm_p.get("PaidLossAmount", 0.0).astype(float).fillna(0.0) + \
                            clm_p.get("ReserveLossAmount", 0.0).astype(float).fillna(0.0)
    loss = clm_p.groupby(AGGREGATION, dropna=False, as_index=False).agg(IncurredLoss=("IncurredLoss","sum"))

    # ---------------------- Assemble KPI ----------------------
    parts = [earn, ret, comg, loss]
    if exp_df is not None:
        parts.append(exp_df)

    kpi = parts[0]
    for p in parts[1:]:
        kpi = kpi.merge(p, on=AGGREGATION, how="outer")

    for c in ["EarnedPremium","PoliciesActive","EligibleForRenewal","Renewals",
              "CommissionWrittenPremium","CommissionAmount","IncurredLoss","UW_Expenses"]:
        if c in kpi.columns:
            kpi[c] = kpi[c].fillna(0.0)

    # Ratios (raw)
    kpi["LossRatio_raw"] = np.where(
        kpi["EarnedPremium"] > MIN_EARNED_FOR_LR,
        kpi["IncurredLoss"] / kpi["EarnedPremium"],
        np.nan
    )
    # Credibility-smoothed LR to avoid 0/1 spikes as the base for mapping
        # ---- replace your LR_base block with this size-capped smoothing ----
    ELR         = 0.65          # expected LR
    ALPHA_BASE  = 25000.0       # smoothing strength
    EP_CAP      = 50000.0       # stop smoothing above this EP

    alpha_eff = np.where(kpi["EarnedPremium"] < EP_CAP, ALPHA_BASE, 0.0)
    kpi["LR_base"] = np.where(
        kpi["EarnedPremium"] > 0,
        (kpi["IncurredLoss"] + ELR * alpha_eff) / (kpi["EarnedPremium"] + alpha_eff),
        np.nan
    ).clip(0, 1)

    if "UW_Expenses" in kpi.columns:
        kpi["ExpenseRatio_raw"] = np.where(
            kpi["EarnedPremium"] > 0,
            kpi["UW_Expenses"] / kpi["EarnedPremium"],
            np.nan
        )

    # Period labels
    kpi["PeriodStart"] = PERIOD_START.date()
    kpi["PeriodEnd"]   = PERIOD_END.date()

    # ----------------- Bounding + flags ----------------
    ret_clip = clip_with_flags(kpi["RetentionRate_raw"], RETENTION_MIN, RETENTION_MAX, "RetentionRate")
    lr_clip  = clip_with_flags(kpi["LossRatio_raw"],      LR_MIN,       LR_MAX,       "LossRatio")
    kpi = pd.concat([kpi, ret_clip, lr_clip], axis=1)

    if "ExpenseRatio_raw" in kpi.columns:
        exp_clip = clip_with_flags(kpi["ExpenseRatio_raw"], 0.0, 1.0, "ExpenseRatio")
        kpi = pd.concat([kpi, exp_clip], axis=1)

    # ------------------- Rewrites vs New (optional) ----------------
    if "TransactionEffectiveDate" in pol.columns and "TransactionType" in pol.columns:
        tx = pol[
            (pol["TransactionEffectiveDate"] >= PERIOD_START) &
            (pol["TransactionEffectiveDate"] <= PERIOD_END)
        ].copy()
        tx["NewCount"] = pol["TransactionType"].astype(str).str.casefold().eq("new").astype(int)
        tx["RewriteCount"] = pol["TransactionType"].astype(str).str.casefold().eq("rewrite").astype(int)
        rw = tx.groupby(AGGREGATION, dropna=False, as_index=False).agg(
            NewCount=("NewCount","sum"), RewriteCount=("RewriteCount","sum")
        )
        kpi = kpi.merge(rw, on=AGGREGATION, how="left")
        kpi[["NewCount","RewriteCount"]] = kpi[["NewCount","RewriteCount"]].fillna(0.0)
        kpi["RewrittenPolicyRatio_raw"] = np.where(
            kpi["NewCount"] > 0, kpi["RewriteCount"] / kpi["NewCount"], np.nan
        )
        rpr_clip = clip_with_flags(kpi["RewrittenPolicyRatio_raw"], 0.0, 1.0, "RewrittenPolicyRatio")
        kpi = pd.concat([kpi, rpr_clip], axis=1)

    # ---------------- SubmissionQuality (optional) ---------------
    if Path(SUBMISSIONS_PATH).exists():
        subs = pd.read_csv(SUBMISSIONS_PATH)
        if "EffDt" in subs.columns:
            subs["EffDt"] = pd.to_datetime(subs["EffDt"], errors="coerce")
        if "AgentCode" in subs.columns and "AgentCode" in kpi.columns:
            subs["AgentCode"] = subs["AgentCode"].astype(str).str.strip()
            kpi["AgentCode"]  = kpi["AgentCode"].astype(str).str.strip()
        status_norm = subs.get("Status", pd.Series("", index=subs.index)).astype(str).str.strip().str.casefold()
        subs["__is_bound__"]  = status_norm.eq("bound").astype(int)
        subs["__is_quoted__"] = status_norm.isin(["quoted","bound"]).astype(int)

        WINDOW_START_OFFSET_DAYS = 60
        def compute_sq_row(row):
            if "AgentCode" not in row or pd.isna(row["PeriodStart"]) or pd.isna(row["PeriodEnd"]):
                return np.nan, 0, 0, 0
            ag = str(row["AgentCode"])
            ps = pd.to_datetime(row["PeriodStart"])
            pe = pd.to_datetime(row["PeriodEnd"])
            start = ps - pd.Timedelta(days=WINDOW_START_OFFSET_DAYS)
            block = subs[(subs["AgentCode"] == ag) & (subs["EffDt"] >= start) & (subs["EffDt"] <= pe)]
            submitted = block["SubmissionId"].nunique() if "SubmissionId" in block.columns else len(block)
            bound     = int(block["__is_bound__"].sum())
            denom = submitted
            val = (bound / denom) if denom else np.nan
            return val, submitted, int(block["__is_quoted__"].sum()), bound

        res = kpi.apply(lambda r: compute_sq_row(r), axis=1, result_type="expand")
        res.columns = ["SubmissionQuality_raw", "Submitted", "Quoted", "Bound"]
        kpi = pd.concat([kpi, res], axis=1)
        sq_clip = clip_with_flags(kpi["SubmissionQuality_raw"], SUBQ_MIN, SUBQ_MAX, "SubmissionQuality")
        kpi = pd.concat([kpi, sq_clip], axis=1)

    # ---------------- Optional ALL row (robust) ----------------
    if INCLUDE_OVERALL_ROW:
        overall = kpi.copy()
        if AGGREGATION:
            for dim in AGGREGATION:
                overall[dim] = "ALL"
        overall = overall.loc[:, ~overall.columns.duplicated(keep="last")].copy()
        has_periods = all(c in overall.columns for c in ["PeriodStart","PeriodEnd"])
        group_keys = (AGGREGATION or []) + (["PeriodStart","PeriodEnd"] if has_periods else [])
        num_base = overall.select_dtypes(include=[np.number]).columns
        bad_suffixes = ("Rate", "Ratio", "_raw", "_was_clipped_low", "_was_clipped_high",
                        "_was_winsor_low", "_was_winsor_high")
        sum_cols = [c for c in num_base if not c.endswith(bad_suffixes)]
        overall = overall.groupby(group_keys, as_index=False)[sum_cols].sum()

        if set(["EligibleForRenewal","Renewals"]).issubset(overall.columns):
            overall["RetentionRate_raw"] = np.where(
                overall["EligibleForRenewal"] > 0,
                overall["Renewals"] / overall["EligibleForRenewal"], np.nan
            )
            overall["RetentionRate"] = overall["RetentionRate_raw"].clip(RETENTION_MIN, RETENTION_MAX)
            overall["RetentionRate_was_clipped_low"]  = (overall["RetentionRate_raw"] < RETENTION_MIN).astype(int)
            overall["RetentionRate_was_clipped_high"] = (overall["RetentionRate_raw"] > RETENTION_MAX).astype(int)

        if set(["CommissionAmount","CommissionWrittenPremium"]).issubset(overall.columns):
            basis = overall["CommissionWrittenPremium"].where(
                overall["CommissionWrittenPremium"].abs() >= MIN_COMMISSION_BASIS, np.nan
            )
            overall["CommissionRatio_raw"] = overall["CommissionAmount"] / basis
            overall["CommissionRatio"] = overall["CommissionRatio_raw"]

        if set(["IncurredLoss","EarnedPremium"]).issubset(overall.columns):
            overall["LossRatio_raw"] = np.where(
                overall["EarnedPremium"] > MIN_EARNED_FOR_LR,
                overall["IncurredLoss"] / overall["EarnedPremium"], np.nan
            )
            overall["LossRatio"] = overall["LossRatio_raw"].clip(LR_MIN, LR_MAX)
            overall["LossRatio_was_clipped_low"]  = (overall["LossRatio_raw"] < LR_MIN).astype(int)
            overall["LossRatio_was_clipped_high"] = (overall["LossRatio_raw"] > LR_MAX).astype(int)

        if set(["UW_Expenses","EarnedPremium"]).issubset(overall.columns):
            overall["ExpenseRatio_raw"] = np.where(
                overall["EarnedPremium"] > 0,
                overall["UW_Expenses"] / overall["EarnedPremium"], np.nan
            )
            overall["ExpenseRatio"] = overall["ExpenseRatio_raw"].clip(0.0, 1.0)
            overall["ExpenseRatio_was_clipped_low"]  = (overall["ExpenseRatio_raw"] < 0.0).astype(int)
            overall["ExpenseRatio_was_clipped_high"] = (overall["ExpenseRatio_raw"] > 1.0).astype(int)

        if set(["RewriteCount","NewCount"]).issubset(overall.columns):
            overall["RewrittenPolicyRatio_raw"] = np.where(
                overall["NewCount"] > 0,
                overall["RewriteCount"] / overall["NewCount"], np.nan
            )
            overall["RewrittenPolicyRatio"] = overall["RewrittenPolicyRatio_raw"].clip(0.0, 1.0)
            overall["RewrittenPolicyRatio_was_clipped_low"]  = (overall["RewrittenPolicyRatio_raw"] < 0.0).astype(int)
            overall["RewrittenPolicyRatio_was_clipped_high"] = (overall["RewrittenPolicyRatio_raw"] > 1.0).astype(int)

        if set(["Bound","Submitted"]).issubset(overall.columns):
            overall["SubmissionQuality_raw"] = np.where(
                overall["Submitted"] > 0, overall["Bound"] / overall["Submitted"], np.nan
            )
            overall["SubmissionQuality"] = overall["SubmissionQuality_raw"].clip(SUBQ_MIN, SUBQ_MAX)
            overall["SubmissionQuality_was_clipped_low"]  = (overall["SubmissionQuality_raw"] < SUBQ_MIN).astype(int)
            overall["SubmissionQuality_was_clipped_high"] = (overall["SubmissionQuality_raw"] > SUBQ_MAX).astype(int)

        for col in [c for c in overall.columns if c.endswith(("Rate","Ratio")) and not c.endswith("_raw")]:
            overall[col] = overall[col].astype(float).round(4)

        overall = overall.loc[:, ~overall.columns.duplicated(keep="last")].copy()
        overall = overall.reindex(columns=kpi.columns, fill_value=np.nan)
        kpi = pd.concat([kpi, overall], ignore_index=True)

    # ---------------- Winsorize $ metrics ----------------
    kpi = winsorize_columns(kpi, [c for c in WINSORIZE_NUM_COLS if c in kpi.columns],
                            lo_q=WINSOR_QUANTS[0], hi_q=WINSOR_QUANTS[1])
    # ---------------- Peer-relative Earned Premium (size de-bias helpers) ----------------
# Choose your peer grouping keys. If you have LOB in kpi, this will group within LOB.
    
    PEER_KEYS = [c for c in ["LOB"] if c in kpi.columns]  # add territory/segment if available

    if PEER_KEYS:
        grp = kpi.groupby(PEER_KEYS, dropna=False)
        # Percentile of EP within peer group
        kpi["EP_percentile_peer"] = grp["EarnedPremium"].rank(pct=True)
        # Concave utility: compress very large EP relative to peer 95th percentile
        p95 = grp["EarnedPremium"].transform(lambda s: s.quantile(0.95))
        kpi["EP_utility_peer"] = (np.log1p(kpi["EarnedPremium"]) / np.log1p(p95.replace(0, np.nan))) \
                                    .clip(0, 1).fillna(0)
        # Handy buckets for fair comparisons
        kpi["EP_decile_peer"] = (kpi["EP_percentile_peer"] * 10).apply(np.ceil).astype("Int64")
    else:
        # If no peer keys available, you can still compute global context (optional)
        kpi["EP_percentile_peer"] = kpi["EarnedPremium"].rank(pct=True)
        p95_global = kpi["EarnedPremium"].quantile(0.95)
        kpi["EP_utility_peer"] = (np.log1p(kpi["EarnedPremium"]) / np.log1p(p95_global if p95_global > 0 else 1)).clip(0, 1)
        kpi["EP_decile_peer"] = (kpi["EP_percentile_peer"] * 10).apply(np.ceil).astype("Int64")

    # ---------------- Rank-preserving banding ----------------
    # Map LossRatio to [0.30, 0.90], RetentionRate to [0.50, 0.95]
    # ---------------- Rank-preserving GAUSSIAN banding ----------------
    # Want bell-shaped LossRatio in [0.30, 0.90], RetentionRate in [0.50, 0.95]
    # ---------------- Rank-preserving GAUSSIAN banding (smooth + tie-safe) ----------------
# Targets: LossRatio in [0.30, 0.90], RetentionRate in [0.50, 0.95]

    # Use LR_base (smoothed) instead of the clipped LossRatio for mapping
    if "LR_base" in kpi.columns:
        kpi["LossRatio_adj"] = _rank_to_truncnorm_smooth(
            kpi["LR_base"], lo=0.30, hi=0.90,
            mu=0.60,          # center the bell where you want it
            sigma=0.10,       # narrower/wider bell; try 0.08–0.12
            jitter_scale=1e-6,
            seed=123
        ).round(4)
        kpi["LRadj_BENEFIT_inverted"] = (1.0 - kpi["LossRatio_adj"]).clip(0, 1).round(4)

    # Retention uses the bounded raw rate; you can also smooth similarly if needed
    if "RetentionRate" in kpi.columns:
        kpi["RetentionRate_adj"] = _rank_to_truncnorm_smooth(
            kpi["RetentionRate"], lo=0.50, hi=0.95,
            mu=0.80,           # many agencies lean high; shift peak right if desired
            sigma=0.08,
            jitter_scale=1e-6,
            seed=321
        ).round(4)

    # ---- Expense amount & raw ratio ----
    # Prefer explicit underwriting expenses if present; else estimate as Commission + OPEX% * EarnedPremium
    if "UW_Expenses" in kpi.columns:
        kpi["ExpenseAmount"] = kpi["UW_Expenses"].astype(float)
    else:
        kpi["ExpenseAmount"] = kpi.get("CommissionAmount", 0.0).astype(float) + OPEX_PCT * kpi["EarnedPremium"].astype(float)

    kpi["ExpenseRatio_raw"] = np.where(
        kpi["EarnedPremium"] > 0,
        kpi["ExpenseAmount"] / kpi["EarnedPremium"],
        np.nan
        )
    kpi["ExpenseRatio_raw"] = kpi["ExpenseRatio_raw"].clip(0.0, 1.0)
    # ---- Expense Ratio adjustment to [0.24, 0.40] as a bell curve ----
    if "ExpenseRatio_raw" in kpi.columns:
        kpi["ExpenseRatio"] = kpi["ExpenseRatio_raw"].clip(0.0, 1.0)  # keep raw bounded for QA
        kpi["ExpenseRatio_adj"] = _rank_to_truncnorm_smooth(
            kpi["ExpenseRatio"], lo=ER_TARGET_LO, hi=ER_TARGET_HI,
            mu=ER_MU, sigma=ER_SIGMA, jitter_scale=1e-6, seed=456
        ).round(4)
        # benefit form for TOPSIS
        kpi["ERadj_BENEFIT_inverted"] = (1.0 - kpi["ExpenseRatio_adj"]).clip(0, 1).round(4)

    # Gaussianized, rank-preserving ER within [0.24, 0.40]
    kpi["ExpenseRatio"] = _rank_to_truncnorm_smooth(
        kpi["ExpenseRatio_raw"], lo=ER_LO, hi=ER_HI, mu=ER_MU, sigma=ER_SIG,
        jitter_scale=1e-6, seed=456
    ).round(4)

    # benefit form for TOPSIS (higher better)
    kpi["ER_BENEFIT_inverted"] = (1.0 - kpi["ExpenseRatio"]).clip(0,1).round(4)
    # ---------------- Final rounding (ratios only) ----------------
    ratio_like = [c for c in kpi.columns if c.endswith("Ratio") or c.endswith("Rate")]
    for c in ["RetentionRate","CommissionRatio","LossRatio","ExpenseRatio",
              "RewrittenPolicyRatio","SubmissionQuality",
              "RetentionRate_adj"]:
        if c in kpi.columns: ratio_like.append(c)
    for col in set(ratio_like):
        kpi[col] = kpi[col].astype(float).round(4)

    # Order columns (dims → period → core KPI → raw/flags last)
    dim_cols = AGGREGATION
    period_cols = ["PeriodStart","PeriodEnd"]
    primary_kpis = [c for c in [
        "RetentionRate_adj","SubmissionQuality","LossRatio_adj","ExpenseRatio",
        "CommissionRatio","RewrittenPolicyRatio",
        "LRadj_BENEFIT_inverted",
        "EarnedPremium","IncurredLoss","CommissionAmount","CommissionWrittenPremium",
        "EligibleForRenewal","Renewals","PoliciesActive","NewCount","RewriteCount",
        "Bound","Quoted","Submitted"
    ] if c in kpi.columns]
    meta_cols = [c for c in kpi.columns if c not in (dim_cols + period_cols + primary_kpis)]
    kpi = kpi[dim_cols + period_cols + primary_kpis + meta_cols]

    # ---------------- Emit full KPI table ----------------
    kpi.to_csv(OUT_KPI_PATH, index=False)
    print(f"✓ Saved cleaned KPI table → {OUT_KPI_PATH}  (rows={len(kpi)})")

    # ---------------- Build a TOPSIS-ready view (uses adjusted metrics) ----------------
    BENEFIT_FEATURES = ["RetentionRate_adj", "SubmissionQuality"]   # higher is better
    EXTRA_BENEFITS   = ["LRadj_BENEFIT_inverted"]                   # already inverted (LossRatio_adj)
    PASSTHROUGH_COLS = [
        "EarnedPremium","IncurredLoss","CommissionAmount","CommissionWrittenPremium",
        "EligibleForRenewal","Renewals","PoliciesActive","NewCount","RewriteCount",
        "Submitted","Quoted","Bound","WindowStartUsed","WindowEndUsed",
        "CommissionRatio","RewrittenPolicyRatio","LossRatio","RetentionRate",
        "LossRatio_adj","RetentionRate_adj","ERadj_BENEFIT_inverted","ExpenseRatio_raw","ExpenseRatio"
    ]

    dim_cols   = AGGREGATION or []
    period_cols = [c for c in ["PeriodStart","PeriodEnd"] if c in kpi.columns]
    topsis_view = kpi[dim_cols + period_cols].copy()

    for f in BENEFIT_FEATURES:
        if f in kpi.columns:
            topsis_view[f] = kpi[f].astype(float).clip(0.0, 1.0)
    for f in EXTRA_BENEFITS:
        if f in kpi.columns:
            topsis_view[f] = kpi[f].astype(float).clip(0.0, 1.0)
    for c in PASSTHROUGH_COLS:
        if c in kpi.columns and c not in topsis_view.columns:
            topsis_view[c] = kpi[c]
    # Also pass peer-relative size fields for slicing/ranking within tiers
    for c in ["EP_percentile_peer","EP_decile_peer","EP_utility_peer"]:
        if c in kpi.columns and c not in topsis_view.columns:
            topsis_view[c] = kpi[c]


    topsis_view.to_csv(OUT_TOPSIS_VIEW_PATH, index=False)
    print(f"✓ Saved TOPSIS-ready view → {OUT_TOPSIS_VIEW_PATH}  (cols={list(topsis_view.columns)})")

    # ---------------- KPI dictionary (quick reference) ----------------
    kpi_dict = {
        "RetentionRate": "Renewals / EligibleForRenewal, bounded to [0.20, 1.00]. Raw value kept in RetentionRate_raw.",
        "RetentionRate_adj": f"Rank-preserving remap of RetentionRate to [{RET_TARGET_LO}, {RET_TARGET_HI}] for realism.",
        "SubmissionQuality": "Bound / Submitted over (PeriodStart-60d, PeriodEnd], bounded to [0.20, 1.00] when submissions.csv present. Raw in SubmissionQuality_raw.",
        "LossRatio": "IncurredLoss / EarnedPremium, bounded to [0.00, 1.00]. Raw in LossRatio_raw.",
        "LossRatio_adj": f"Rank-preserving remap of LossRatio to [{LR_TARGET_LO}, {LR_TARGET_HI}] for realism.",
        "ExpenseRatio": "UW_Expenses / EarnedPremium if available, bounded to [0.00, 1.00]. Raw in ExpenseRatio_raw.",
        "CommissionRatio": "CommissionAmount / CommissionWrittenPremium with a $100 min basis guard. Raw in CommissionRatio_raw.",
        "RewrittenPolicyRatio": "RewriteCount / NewCount, bounded to [0.00, 1.00]. Raw in RewrittenPolicyRatio_raw.",
        "Winsorization": f"Gentle winsorization at quantiles {WINSOR_QUANTS} for {WINSORIZE_NUM_COLS} to reduce extreme tails.",
    }
    print("\nKPI dictionary:")
    for k, v in kpi_dict.items():
        print(f" - {k}: {v}")


if __name__ == "__main__":
    main()


✓ Saved cleaned KPI table → kpi_clean.csv  (rows=301)
✓ Saved TOPSIS-ready view → kpi_topsis_ready.csv  (cols=['AgentCode', 'PeriodStart', 'PeriodEnd', 'RetentionRate_adj', 'SubmissionQuality', 'LRadj_BENEFIT_inverted', 'EarnedPremium', 'IncurredLoss', 'CommissionAmount', 'CommissionWrittenPremium', 'EligibleForRenewal', 'Renewals', 'PoliciesActive', 'NewCount', 'RewriteCount', 'Submitted', 'Quoted', 'Bound', 'RewrittenPolicyRatio', 'LossRatio', 'RetentionRate', 'LossRatio_adj', 'ERadj_BENEFIT_inverted', 'ExpenseRatio_raw', 'EP_percentile_peer', 'EP_decile_peer', 'EP_utility_peer'])

KPI dictionary:
 - RetentionRate: Renewals / EligibleForRenewal, bounded to [0.20, 1.00]. Raw value kept in RetentionRate_raw.
 - RetentionRate_adj: Rank-preserving remap of RetentionRate to [0.5, 0.95] for realism.
 - SubmissionQuality: Bound / Submitted over (PeriodStart-60d, PeriodEnd], bounded to [0.20, 1.00] when submissions.csv present. Raw in SubmissionQuality_raw.
 - LossRatio: IncurredLoss / Earne